# 🎬 Universal AI Reel Generator (Audio-to-Video Pipeline)

Welcome to the **Universal AI Reel Generator**! This is a 100% generic, end-to-end automated pipeline capable of generating any type of high-retention vertical video (Instagram Reels, YouTube Shorts, or TikTok) solely on the basis of a single audio recording.

**You only need to provide an audio file**—the AI automatically handles everything else:
* **Studio Audio Mastering & Alignment:** Removes background noise, enhances vocal clarity, and extracts exact millisecond word-level timestamps using **Resemble Enhance** and **WhisperX**.
* **Interactive Waveform Timeline:** Features a built-in CapCut-style waveform editor to easily trim mistakes and auto-sync transcripts.
* **Context-Aware Visual Direction:** Uses **Qwen2.5-7B** to analyze your spoken words and dynamically design literal, non-repetitive visual storyboards, topic badges, and takeaways.
* **Ultra-Fast AI Scene Generation:** Generates vibrant, high-contrast visual assets for every 3-second segment using **LCM-DreamShaper**.
* **Split-Screen Video Compositing:** Renders a 1080x1920 vertical video combining Ken Burns animated visuals, live word-by-word highlighted karaoke captions, and a voice-reactive corner presenter avatar.

# ☁️ Step 1: Persistent Google Drive Environment & Model Cache Manager

This cell initializes the **Zero-Redownload Persistent Architecture** for the AI Reel Generator. Instead of reinstalling heavy dependencies and re-downloading multi-gigabyte AI models every time the Colab runtime resets, this setup locks the entire software stack permanently inside Google Drive.

### ⚡ What This Setup Does:
* **Persistent Package Environment (`Colab_AI_Env`):** Uses a smart marker check (`.all_packages_installed_v1`) to load all required Python libraries directly from Google Drive in under **1 second**, running installations only on the very first run.
* **Permanent AI Model Vault (`AI_Models_Cache`):** Maps HuggingFace, PyTorch, and Coqui TTS cache paths directly to Google Drive, ensuring heavy models (**Qwen2.5-7B**, **LCM-DreamShaper**, and **Resemble-Enhance**) are downloaded only once and loaded offline thereafter.
* **Auto-Backup & Self-Healing Verification:** Automatically backs up generated audio and timestamp files (`step1_clean.wav`, `raw_words.json`, `final_flawless_voice.wav`, `synced_final_words.json`) to your project folder and verifies all packages and model weights before starting the pipeline.

In [ ]:
# ==============================================================================
# 👑 MASTER CELL 1: ALL INSTALLATIONS (APT + UV), DRIVE MODELS, AVATAR & IMPORTS
# ==============================================================================
import base64
from base64 import b64encode
import gc
import glob
import json
import math
import operator
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys
import warnings
from google.colab import drive, output
from IPython.display import Audio, HTML, display

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ==============================================================================
# 🎯 0. USER INPUTS, OUTPUTS & PATH CONFIGURATION (Sirf Yahan Se Change Karein!)
# ==============================================================================

# 👉 [1. PROJECT & CACHE FOLDERS IN GOOGLE DRIVE]:
PROJECT_FOLDER_MAIN = "/content/drive/MyDrive/Video Editing"
PROJECT_FOLDER_FALLBACK = "/content/drive/MyDrive/Colab Notebooks/Video Editing"
DRIVE_ENV_DIR = "/content/drive/MyDrive/Colab_AI_Env"
DRIVE_MODELS_DIR = "/content/drive/MyDrive/AI_Models_Cache"

# 👉 [2. PRESENTER PHOTO INPUT & STUDIO AVATAR OUTPUTS]:
# Agar aap koi nayi apni photo lagana chahein to uska naam yahan likhein ("" chhorne par folder ki pehli .jpeg/.jpg utha lega)
INPUT_RAW_PHOTO_NAME = "WhatsApp Image 2026-11-13 at 9.49.28 PM.jpeg"

# Output Studio Avatar filenames (Drive mein kis naam se save ho aur Colab mein kis naam se)
OUTPUT_DRIVE_AVATAR_NAME = "ahsan_studio_avatar.png"
OUTPUT_LOCAL_AVATAR_NAME = "studio_presenter.png"

# False = Agar avatar pehle se Drive mein bana para hai to 0s mein load karega
# True  = Nayi photo se zabardasti dobara background remove kar ke naya Studio Avatar banayega
FORCE_REGEN_AVATAR = False

# 👉 [3. STUDIO AVATAR CROP & STYLING SETTINGS]:
AVATAR_RESOLUTION = 768  # Output avatar image size (768x768)
AVATAR_CROP_BOX_RATIOS = (
    0.18,
    0.08,
    0.78,
    0.88,
)  # (left, top, right, bottom) crop %
STUDIO_BG_RGBA = (12, 16, 28, 255)  # Dark Cyber-Tech Navy Background
STUDIO_GLOW_RGBA = (0, 165, 235, 95)  # Cyan-Blue Radial Backlight Glow

# 🔒 [4. FIXED AI MODEL SUBFOLDERS IN GOOGLE DRIVE (DO NOT CHANGE)]:
QWEN_MODEL_REPO = "Qwen/Qwen2.5-7B-Instruct"
LCM_MODEL_REPO = "SimianLuo/LCM_Dreamshaper_v7"
RESEMBLE_MODEL_REPO = "ResembleAI/resemble-enhance"
WHISPER_MODEL_NAME = "large-v3"
ALIGNMENT_LANG_CODE = "ur"

# ==============================================================================
# 1. MOUNT GOOGLE DRIVE & INITIALIZE DIRECTORIES (DO NOT CHANGE BELOW)
# ==============================================================================
if not os.path.exists("/content/drive/MyDrive"):
  drive.mount("/content/drive")

DRIVE_ENV = DRIVE_ENV_DIR
DRIVE_CACHE = DRIVE_MODELS_DIR
PROJECT_DIR = (
    PROJECT_FOLDER_MAIN
    if os.path.exists(PROJECT_FOLDER_MAIN)
    else PROJECT_FOLDER_FALLBACK
)

QWEN_DIR = f"{DRIVE_CACHE}/Qwen2.5-7B-Instruct"
LCM_DIR = f"{DRIVE_CACHE}/LCM_Dreamshaper_v7"
RESEMBLE_DIR = f"{DRIVE_CACHE}/resemble-enhance"
WHISPER_DIR = f"{DRIVE_CACHE}/whisperx"
WAV2VEC_DIR = f"{DRIVE_CACHE}/wav2vec2"
U2NET_DIR = f"{DRIVE_CACHE}/u2net"

for d in [
    DRIVE_ENV,
    DRIVE_CACHE,
    PROJECT_DIR,
    QWEN_DIR,
    LCM_DIR,
    RESEMBLE_DIR,
    WHISPER_DIR,
    WAV2VEC_DIR,
    U2NET_DIR,
]:
  os.makedirs(d, exist_ok=True)

if DRIVE_ENV not in sys.path:
  sys.path.insert(0, DRIVE_ENV)

os.environ["HF_HOME"] = f"{DRIVE_CACHE}/huggingface"
os.environ["TORCH_HOME"] = f"{DRIVE_CACHE}/torch"
os.environ["U2NET_HOME"] = U2NET_DIR

# ==============================================================================
# 2. LINUX SYSTEM FONTS & URDU RAQM SHAPING LIBRARIES
# ==============================================================================
print("🔤 [1/5] Installing Linux Urdu/Emoji Fonts & HarfBuzz/Raqm...")
!apt-get update -qq && apt-get install -y -qq fonts-noto-core fonts-noto-color-emoji fonts-hosny-amiri fonts-dejavu-core libraqm0 libfribidi0 libharfbuzz0b >/dev/null 2>&1

# ==============================================================================
# 3. ALL PYTHON PACKAGES VIA UV (PINNED VERSIONS FOR ZERO CONFLICTS)
# ==============================================================================
MARKER_FILE = os.path.join(DRIVE_ENV, ".master_packages_locked_v3")

if os.path.exists(MARKER_FILE):
  print(
      "⚡ [2/5] All Python Packages already saved in Google Drive! (Skipping"
      " Drive install - 0s)"
  )
else:
  print(
      "📦 [2/5] Installing ALL Python packages into Google Drive via uv"
      " (One-Time Setup)..."
  )
  !pip install -q uv
  !uv pip install --target="{DRIVE_ENV}" \
      faster-whisper whisperx pydub einops ptflops deepspeed \
      accelerate bitsandbytes celluloid librosa matplotlib pandas \
      invisible-watermark safetensors beautifulsoup4 requests rich resampy \
      scipy soundfile tqdm OmegaConf python-docx pillow \
      rembg onnxruntime-gpu arabic-reshaper python-bidi \
      "huggingface-hub==0.29.1" "transformers==4.48.3" "diffusers==0.31.0"
  !uv pip install --target="{DRIVE_ENV}" --no-deps resemble-enhance

  with open(MARKER_FILE, "w") as f:
    f.write("OK")
  print("✅ [2/5] All Python packages permanently locked in Google Drive!")

# Sync system packages with exact pinned versions so /usr/local/lib never clashes
!pip install -q uv
!uv pip install --system -q faster-whisper whisperx pydub einops ptflops deepspeed accelerate bitsandbytes celluloid librosa matplotlib pandas invisible-watermark safetensors beautifulsoup4 requests rich resampy scipy soundfile tqdm OmegaConf python-docx rembg onnxruntime-gpu arabic-reshaper python-bidi "huggingface-hub==0.29.1" "transformers==4.48.3" "diffusers==0.31.0"
!uv pip install --system -q --no-deps resemble-enhance

for mod_name in list(sys.modules.keys()):
  if mod_name.startswith(("transformers", "diffusers", "huggingface_hub")):
    del sys.modules[mod_name]

# ==============================================================================
# 4. LOAD & VERIFY ALL IMPORTS + GLOBAL COMPATIBILITY PATCHES
# ==============================================================================
print("🧠 [3/5] Loading All Libraries & Applying Compatibility Patches...")
import arabic_reshaper
from bidi.algorithm import get_display
import diffusers
from diffusers import DiffusionPipeline
import docx
from huggingface_hub import snapshot_download
import librosa
import numpy as np
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter, ImageFont, features
from pydub import AudioSegment
from rembg import remove
from resemble_enhance.enhancer.inference import enhance
import resemble_enhance.enhancer.lcfm.cfm as cfm_mod
from scipy.io import wavfile
import scipy.optimize
import scipy.optimize._minpack_py as _minpack
import soundfile as sf
import torch
import torchaudio
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import transformers.utils
import whisperx

if not hasattr(transformers.utils, "download_url"):
  transformers.utils.download_url = lambda url, *a, **kw: url


# Patch A: Recursion-proof scipy.optimize.fsolve for Resemble-Enhance
def _safe_fsolve(*args, **kwargs):
  res = _minpack.fsolve(*args, **kwargs)
  return (
      res.item()
      if hasattr(res, "ndim") and res.ndim == 1 and res.size == 1
      else res
  )


scipy.optimize.fsolve = _safe_fsolve
cfm_mod.scipy.optimize.fsolve = _safe_fsolve

# Patch B: PyTorch 2.6+ weights_only=False for DeepSpeed Checkpoints
if not hasattr(torch, "_orig_load_resemble"):
  torch._orig_load_resemble = torch.load


def _safe_torch_load(*args, **kwargs):
  kwargs["weights_only"] = False
  return torch._orig_load_resemble(*args, **kwargs)


torch.load = _safe_torch_load

device = "cuda" if torch.cuda.is_available() else "cpu"
print(
    f"   ✅ Imports Ready | GPU Active: {torch.cuda.is_available()}"
    f" ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})"
)

# ==============================================================================
# 5. SMART DRIVE MODEL VAULT (CHECKS ALL 5 MODELS IN FIXED FOLDERS)
# ==============================================================================
print("\n🔍 [4/5] Verifying All 5 AI Models in Google Drive...")

# (A) Qwen2.5-7B-Instruct
if len(glob.glob(f"{QWEN_DIR}/*.safetensors")) >= 4:
  print(f"   ✅ [MODEL READY IN DRIVE] Qwen2.5-7B-Instruct -> {QWEN_DIR}")
else:
  print(f"   📥 [DOWNLOADING ONCE] {QWEN_MODEL_REPO} to Google Drive...")
  snapshot_download(
      repo_id=QWEN_MODEL_REPO, local_dir=QWEN_DIR, local_dir_use_symlinks=False
  )
  print(f"   ✅ [MODEL SAVED IN DRIVE] Qwen2.5-7B-Instruct -> {QWEN_DIR}")

# (B) LCM-DreamShaper-v7 (Clean FP16 Safetensors only)
if os.path.exists(
    f"{LCM_DIR}/unet/diffusion_pytorch_model.safetensors"
) and os.path.exists(f"{LCM_DIR}/vae/diffusion_pytorch_model.safetensors"):
  print(f"   ✅ [MODEL READY IN DRIVE] LCM-DreamShaper-v7  -> {LCM_DIR}")
else:
  print(f"   📥 [DOWNLOADING ONCE] {LCM_MODEL_REPO} to Google Drive...")
  snapshot_download(
      repo_id=LCM_MODEL_REPO,
      local_dir=LCM_DIR,
      local_dir_use_symlinks=False,
      allow_patterns=[
          "*.json",
          "*.txt",
          "unet/*.safetensors",
          "vae/*.safetensors",
          "text_encoder/*.safetensors",
          "feature_extractor/*",
          "scheduler/*",
          "tokenizer/*",
      ],
  )
  print(f"   ✅ [MODEL SAVED IN DRIVE] LCM-DreamShaper-v7  -> {LCM_DIR}")

# (C) Resemble-Enhance (With corruption check - 524288000 bytes = 500 MB)
resemble_pt = (
    f"{RESEMBLE_DIR}/enhancer_stage2/ds/G/default/mp_rank_00_model_states.pt"
)
resemble_valid = False
if os.path.exists(resemble_pt) and os.path.getsize(resemble_pt) > 524288000:
  try:
    _ = torch.load(resemble_pt, map_location="cpu")
    resemble_valid = True
  except Exception:
    shutil.rmtree(RESEMBLE_DIR, ignore_errors=True)
    os.makedirs(RESEMBLE_DIR, exist_ok=True)

if resemble_valid:
  print(f"   ✅ [MODEL READY IN DRIVE] Resemble-Enhance    -> {RESEMBLE_DIR}")
else:
  print(
      f"   📥 [DOWNLOADING ONCE] {RESEMBLE_MODEL_REPO} (713MB) to Google"
      " Drive..."
  )
  snapshot_download(
      repo_id=RESEMBLE_MODEL_REPO,
      local_dir=RESEMBLE_DIR,
      local_dir_use_symlinks=False,
      allow_patterns=["enhancer_stage2/*"],
      force_download=True,
  )
  print(f"   ✅ [MODEL SAVED IN DRIVE] Resemble-Enhance    -> {RESEMBLE_DIR}")

# (D) WhisperX Faster-Whisper Large-v3 (~3.09 GB in WHISPER_DIR)
whisper_bins = [
    p
    for p in glob.glob(f"{WHISPER_DIR}/**/model.bin", recursive=True)
    if os.path.getsize(p) > 2000000000
]
if whisper_bins:
  print(f"   ✅ [MODEL READY IN DRIVE] Whisper-Large-v3    -> {WHISPER_DIR}")
else:
  print("   📥 [DOWNLOADING ONCE] Whisper-Large-v3 (3.09GB) to Google Drive...")
  _w_model = whisperx.load_model(
      WHISPER_MODEL_NAME,
      device="cpu",
      compute_type="float32",
      language=ALIGNMENT_LANG_CODE,
      download_root=WHISPER_DIR,
  )
  del _w_model
  gc.collect()
  print(f"   ✅ [MODEL SAVED IN DRIVE] Whisper-Large-v3    -> {WHISPER_DIR}")

# (E) Wav2Vec2 Urdu Phoneme Alignment Model (in WAV2VEC_DIR)
wav2vec_bins = (
    glob.glob(f"{WAV2VEC_DIR}/**/*.bin", recursive=True)
    + glob.glob(f"{WAV2VEC_DIR}/**/*.safetensors", recursive=True)
    + glob.glob(f"{WAV2VEC_DIR}/**/*.pt", recursive=True)
)
if wav2vec_bins:
  print(f"   ✅ [MODEL READY IN DRIVE] Wav2Vec2-Urdu       -> {WAV2VEC_DIR}")
else:
  print(
      "   📥 [DOWNLOADING ONCE] Wav2Vec2-Urdu Alignment Model to Google"
      " Drive..."
  )
  _align_m, _ = whisperx.load_align_model(
      language_code=ALIGNMENT_LANG_CODE, device="cpu", model_dir=WAV2VEC_DIR
  )
  del _align_m
  gc.collect()
  print(f"   ✅ [MODEL SAVED IN DRIVE] Wav2Vec2-Urdu       -> {WAV2VEC_DIR}")

# ==============================================================================
# 6. STUDIO PRESENTER AVATAR (USES TOP INPUT/OUTPUT VARIABLES)
# ==============================================================================
print("\n🎨 [5/5] Verifying Cyber-Tech Studio Presenter Avatar...")
drive_avatar_path = os.path.join(PROJECT_DIR, OUTPUT_DRIVE_AVATAR_NAME)

if not FORCE_REGEN_AVATAR and os.path.exists(drive_avatar_path):
  shutil.copy(drive_avatar_path, OUTPUT_LOCAL_AVATAR_NAME)
  print(
      f"   ✅ [AVATAR READY IN DRIVE] Loaded '{OUTPUT_DRIVE_AVATAR_NAME}' ->"
      f" '{OUTPUT_LOCAL_AVATAR_NAME}'"
  )
else:
  # Check user-specified photo name first, otherwise fallback to any .jpeg/.jpg in PROJECT_DIR
  explicit_photo = os.path.join(PROJECT_DIR, INPUT_RAW_PHOTO_NAME)
  if INPUT_RAW_PHOTO_NAME and os.path.exists(explicit_photo):
    selected_photo = explicit_photo
  elif INPUT_RAW_PHOTO_NAME and os.path.exists(INPUT_RAW_PHOTO_NAME):
    selected_photo = INPUT_RAW_PHOTO_NAME
  else:
    photos = (
        glob.glob(f"{PROJECT_DIR}/*.jpeg")
        + glob.glob(f"{PROJECT_DIR}/*.jpg")
        + glob.glob("/content/*.jpeg")
    )
    selected_photo = photos[0] if photos else None

  if selected_photo:
    print(f"   📸 Input Raw Photo  : {selected_photo}")
    print("   🎨 Removing background & compositing Studio Avatar...")
    orig = Image.open(selected_photo).convert("RGBA")
    w, h = orig.size
    l_r, t_r, r_r, b_r = AVATAR_CROP_BOX_RATIOS
    crop_coords = (
        int(operator.mul(w, l_r)),
        int(operator.mul(h, t_r)),
        int(operator.mul(w, r_r)),
        int(operator.mul(h, b_r)),
    )
    cropped = orig.crop(crop_coords).resize(
        (AVATAR_RESOLUTION, AVATAR_RESOLUTION), Image.Resampling.LANCZOS
    )

    fg = remove(cropped)
    bg = Image.new(
        "RGBA", (AVATAR_RESOLUTION, AVATAR_RESOLUTION), STUDIO_BG_RGBA
    )
    glow = Image.new(
        "RGBA", (AVATAR_RESOLUTION, AVATAR_RESOLUTION), (0, 0, 0, 0)
    )
    ImageDraw.Draw(glow).ellipse((140, 120, 628, 620), fill=STUDIO_GLOW_RGBA)
    glow = glow.filter(ImageFilter.GaussianBlur(radius=95))

    final_av = Image.alpha_composite(
        Image.alpha_composite(bg, glow), fg
    ).convert("RGB")
    final_av.save(OUTPUT_LOCAL_AVATAR_NAME, quality=98)
    shutil.copy(OUTPUT_LOCAL_AVATAR_NAME, drive_avatar_path)
    print(
        f"   ✅ [AVATAR SAVED] Local: '{OUTPUT_LOCAL_AVATAR_NAME}' | Drive:"
        f" '{drive_avatar_path}'"
    )
  else:
    print(
        f"   ⚠️ Raw photo '{INPUT_RAW_PHOTO_NAME}' nahi mili! '{PROJECT_DIR}'"
        " mein photo rakhein."
    )

print("\n🎉 EVERYTHING IS INSTALLED, CACHED IN DRIVE & LOADED IN MEMORY!")

# 🎙️ Step 2: Studio Audio Enhancement & Word-Level Timestamp Alignment

This stage takes your raw input audio file and processes it through a two-stage AI audio pipeline to prepare both studio-grade sound and exact millisecond synchronization data for the video editor and visual generator.

### ⚡ What This Step Does:
* **AI Studio Voice Mastering (`Resemble-Enhance`):** Strips out background room noise, echo, and microphone hiss, reconstructing vocal frequencies to deliver crisp, broadcast-quality studio audio (`step1_clean.wav`). Includes a recursion-proof `scipy.optimize.fsolve` patch for Python 3.13 / NumPy 2.0 compatibility.
* **Bilingual Speech Recognition (`WhisperX Large-v3`):** Transcribes Urdu/Hindi speech alongside technical English terminology with high accuracy using custom ASR prompting.
* **Phoneme-Level Millisecond Alignment (`Wav2Vec2`):** Maps every spoken word to its exact start and end millisecond (`raw_words.json`), powering frame-accurate waveform editing, dynamic scene transitions, and live word-by-word karaoke captions.
* **Smart Drive Caching:** Automatically backs up the cleaned audio and word timestamps to Google Drive, allowing instant reload (`0 seconds`) on future runs unless `force_rerun=True` is specified.

In [ ]:
# ==============================================================================
# 🎙️ STEP 2 (v4): STUDIO AUDIO + WORD TIMESTAMPS (auto-verified, with fallback)
# ==============================================================================
import gc
import glob
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys
import unicodedata
import warnings

warnings.filterwarnings("ignore")

import numpy as np
from pydub import AudioSegment
import resemble_enhance.enhancer.lcfm.cfm as cfm_mod
from resemble_enhance.enhancer.inference import enhance
import scipy.optimize
import scipy.optimize._minpack_py as _minpack
import soundfile as sf
import torch
import torch.serialization
import torchaudio
import whisperx

# ==============================================================================
# 🎯 0. USER SETTINGS
# ==============================================================================
INPUT_AUDIO_FILENAME = "cdn.mp3"
FORCE_RERUN = True
AUDIO_LANGUAGE = "ur"

# Prompt Urdu script mein (Whisper Urdu script mein likhe, aligner ko wohi chahiye)
WHISPER_INITIAL_PROMPT = "یہ ایک ٹیکنیکل ویڈیو ہے جس میں اردو اور انگریزی الفاظ استعمال ہوتے ہیں۔"

# "auto"           = pehle Wav2Vec2 aligner, agar timestamps audio se match na karein
#                    to khud Whisper ke native word timestamps par switch
# "wav2vec"        = sirf Wav2Vec2 aligner
# "whisper_native" = sirf Whisper ke apne word timestamps
ALIGN_METHOD = "auto"

# True = ASR + alignment original audio par (timing enhanced audio jaisi hi rehti hai)
ALIGN_ON_RAW = True

OUTPUT_CLEAN_WAV = "step1_clean.wav"
OUTPUT_WORDS_JSON = "raw_words.json"
RAW_MASTER_WAV = "step0_raw_master.wav"
CACHE_VERSION = 4

# ==============================================================================
# 1. PATHS
# ==============================================================================
DRIVE_ENV = "/content/drive/MyDrive/Colab_AI_Env"
DRIVE_CACHE = "/content/drive/MyDrive/AI_Models_Cache"
if DRIVE_ENV not in sys.path:
  sys.path.insert(0, DRIVE_ENV)

PROJECT_DIR = "/content/drive/MyDrive/Video Editing"
if not Path(PROJECT_DIR).exists():
  PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/Video Editing"

RESEMBLE_DIR = f"{DRIVE_CACHE}/resemble-enhance"
WHISPER_DIR = f"{DRIVE_CACHE}/whisperx"
WAV2VEC_DIR = f"{DRIVE_CACHE}/wav2vec2"
HF_BASE = f"{DRIVE_CACHE}/huggingface"

os.makedirs(WHISPER_DIR, exist_ok=True)
os.makedirs(WAV2VEC_DIR, exist_ok=True)
os.environ["HF_HOME"] = HF_BASE
os.environ["TORCH_HOME"] = f"{DRIVE_CACHE}/torch"

MEDIA_EXT = (".mp3", ".wav", ".m4a", ".ogg", ".mpeg", ".mp4", ".mov", ".mkv", ".webm")


def locate_raw_audio(filename: str) -> str:
  candidates = [
      filename,
      os.path.join("/content", filename),
      os.path.join(PROJECT_DIR, filename),
      os.path.join("/content/drive/MyDrive", filename),
  ]
  for c in candidates:
    if os.path.exists(c):
      return c
  stem = Path(filename).stem.lower()
  for sdir in ["/content", PROJECT_DIR, "/content/drive/MyDrive"]:
    if os.path.exists(sdir):
      for f in os.listdir(sdir):
        if f.lower() == filename.lower() or (
            Path(f).stem.lower() == stem and f.lower().endswith(MEDIA_EXT)
        ):
          return os.path.join(sdir, f)
  raise FileNotFoundError(f"❌ '{filename}' nahi mili!")


RAW_AUDIO_PATH = locate_raw_audio(INPUT_AUDIO_FILENAME)


def extract_master_wav(src: str, dst: str = RAW_MASTER_WAV) -> str:
  """Kisi bhi audio/video -> mono 44.1k PCM wav (timeline start 0 se aligned)."""
  cmd = [
      "ffmpeg", "-y", "-loglevel", "error", "-i", src, "-vn", "-ac", "1",
      "-ar", "44100", "-af", "aresample=async=1:first_pts=0",
      "-c:a", "pcm_s16le", dst,
  ]
  subprocess.run(cmd, check=True)
  return dst


# ==============================================================================
# 2. PATCHES (scipy / torch.load / alignment loader)
# ==============================================================================
def _safe_fsolve(*args, **kwargs):
  res = _minpack.fsolve(*args, **kwargs)
  return (
      res.item()
      if hasattr(res, "ndim") and res.ndim == 1 and res.size == 1
      else res
  )


scipy.optimize.fsolve = _safe_fsolve
cfm_mod.scipy.optimize.fsolve = _safe_fsolve

if not hasattr(torch, "_orig_load_resemble"):
  torch._orig_load_resemble = torch.load
if not hasattr(torch.serialization, "_orig_load_resemble"):
  torch.serialization._orig_load_resemble = torch.serialization.load


def _safe_torch_load(*args, **kwargs):
  kwargs["weights_only"] = False
  return torch._orig_load_resemble(*args, **kwargs)


def _safe_ser_load(*args, **kwargs):
  kwargs["weights_only"] = False
  return torch.serialization._orig_load_resemble(*args, **kwargs)


torch.load = _safe_torch_load
torch.serialization.load = _safe_ser_load

import whisperx.alignment as wx_align
from transformers import AutoFeatureExtractor, AutoTokenizer, Wav2Vec2ForCTC


class AlignmentMetadata(dict):

  def __getattr__(self, key):
    if key in self:
      return self[key]
    raise AttributeError(key)

  def __setattr__(self, key, value):
    self[key] = value


def _patched_load_align_model(
    language_code, device, model_name=None, model_dir=None,
    model_cache_only=False,
):
  if model_name is None:
    if language_code == "ur":
      model_name = "kingabzpro/wav2vec2-large-xls-r-300m-Urdu"
    else:
      model_name = f"jonatasgrosman/wav2vec2-large-xlsr-53-{language_code}"
  print(f"   📂 Loading Alignment Model: {model_name}...")
  feature_extractor = AutoFeatureExtractor.from_pretrained(
      model_name, cache_dir=model_dir, local_files_only=model_cache_only)
  tokenizer = AutoTokenizer.from_pretrained(
      model_name, cache_dir=model_dir, local_files_only=model_cache_only)
  align_model = Wav2Vec2ForCTC.from_pretrained(
      model_name, cache_dir=model_dir, local_files_only=model_cache_only)
  align_model = align_model.to(device)

  # 🔴 ASAL BUG FIX: WhisperX dictionary ke keys LOWERCASE expect karta hai
  # (blank token ko '[pad]' naam se dhoondta hai). Pehle '[PAD]' pass ho raha
  # tha => blank galat token ban jata tha => aligner saare lafz shuru ke
  # thode seconds mein thoons deta tha.
  raw_vocab = dict(tokenizer.get_vocab())
  dictionary = {str(k).lower(): int(v) for k, v in raw_vocab.items()}
  if tokenizer.pad_token_id is not None:
    dictionary["[pad]"] = int(tokenizer.pad_token_id)
  if "|" not in dictionary and " " in dictionary:
    dictionary["|"] = dictionary[" "]
  print(
      f"   🔧 Aligner vocab: {len(dictionary)} tokens | blank([pad])="
      f"{dictionary.get('[pad]')} | word-delimiter '|'={dictionary.get('|')}"
  )

  metadata = AlignmentMetadata({
      "dictionary": dictionary,
      "language": language_code,
      "type": "huggingface",
      "tokenizer": tokenizer,
      "feature_extractor": feature_extractor,
  })
  return align_model, metadata


wx_align.load_align_model = _patched_load_align_model
whisperx.load_align_model = _patched_load_align_model


def load_audio_tensor_clean(audio_path: str):
  seg = AudioSegment.from_file(str(audio_path)).set_channels(1)
  sr = seg.frame_rate
  samples = np.array(seg.get_array_of_samples(), dtype=np.float32)
  samples = samples / float(1 << (8 * seg.sample_width - 1))
  return torch.from_numpy(samples).float(), sr


def _patched_torchaudio_load(filepath, *args, **kwargs):
  wav_1d, sr = load_audio_tensor_clean(str(filepath))
  return wav_1d.unsqueeze(0), sr


def _patched_torchaudio_save(filepath, src, sample_rate, *args, **kwargs):
  wav_np = src.detach().cpu().numpy()
  if wav_np.ndim > 1 and wav_np.shape[0] == 1:
    wav_np = wav_np[0]
  elif wav_np.ndim > 1:
    wav_np = wav_np.T
  sf.write(str(filepath), wav_np, sample_rate)


torchaudio.load = _patched_torchaudio_load
torchaudio.save = _patched_torchaudio_save

device = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if device == "cuda" else "int8"


def get_resemble_run_dir() -> Path:
  return Path(RESEMBLE_DIR) / "enhancer_stage2"


def get_whisper_model_path() -> str:
  bins = [
      p for p in glob.glob(f"{WHISPER_DIR}/**/model.bin", recursive=True)
      if os.path.getsize(p) > 2000000000
  ]
  if bins:
    return str(Path(bins[0]).parent)
  return "large-v3"


# ==============================================================================
# 3. TEXT NORMALIZATION
# ==============================================================================
_DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED\u0640]")
_CHAR_MAP = {
    "ك": "ک", "ي": "ی", "ى": "ی", "ې": "ی", "ۍ": "ی",
    "ە": "ہ", "ة": "ہ", "ۀ": "ہ", "أ": "ا", "إ": "ا", "ٱ": "ا",
}


def norm_token(tok: str, vocab_lower: set) -> str:
  t = unicodedata.normalize("NFC", tok)
  t = "".join(_CHAR_MAP.get(c, c) for c in t)
  t = _DIACRITICS.sub("", t)
  t = re.sub(r"[^\w]", "", t)
  out = []
  for c in t:
    if "\u0600" <= c <= "\u06FF" and c.lower() not in vocab_lower:
      continue
    out.append(c)
  return "".join(out)


# ==============================================================================
# 4. WORD LIST BUILDER + QUALITY CHECK
# ==============================================================================
def build_words(aligned_segments, audio_dur):
  flat = []
  for seg in aligned_segments:
    s0, s1 = float(seg["start"]), float(seg["end"])
    words = seg.get("words") or []
    if not words:
      words = [{"word": t} for t in seg.get("text", "").split()]
    for w in words:
      has = ("start" in w and "end" in w and w["start"] is not None
             and w["end"] is not None and not np.isnan(w["start"])
             and not np.isnan(w["end"]))
      flat.append({
          "word": w["word"],
          "start": float(w["start"]) if has else None,
          "end": float(w["end"]) if has else None,
          "seg_start": s0, "seg_end": s1, "est": not has,
      })

  n = len(flat)
  i = 0
  while i < n:
    if flat[i]["start"] is not None:
      i += 1
      continue
    j = i
    while j + 1 < n and flat[j + 1]["start"] is None:
      j += 1
    left = flat[i - 1]["end"] if i > 0 else flat[i]["seg_start"]
    right = flat[j + 1]["start"] if j + 1 < n else flat[j]["seg_end"]
    run = flat[i:j + 1]
    if right <= left:
      right = left + 0.12 * len(run)
    weights = [max(1, len(r["word"])) for r in run]
    total = float(sum(weights))
    cur = left
    for r, wt in zip(run, weights):
      d = (right - left) * wt / total
      r["start"], r["end"] = cur, cur + d
      cur += d
    i = j + 1

  out, prev_end = [], 0.0
  for w in flat:
    s = max(w["start"], prev_end)
    e = max(w["end"], s + 0.03)
    if audio_dur:
      e = min(e, audio_dur)
      s = min(s, e - 0.01)
    prev_end = e
    out.append({
        "id": len(out), "word": w["word"],
        "start": round(s, 3), "end": round(e, 3), "est": w["est"],
    })
  return out


def alignment_quality(words, x, sr):
  """Audio ki awaaz (energy) se match: kya words wahin hain jahan awaaz hai?
  coverage  = awaaz wale hisson ka kitna % words se cover hai
  precision = kitne % words awaaz wali jagah par hain
  peak_wps  = kisi bhi 3s window mein max lafz/sec (>8.5 => thoonsa hua)"""
  fl = int(0.05 * sr)
  n = len(x) // fl
  bad = {"coverage": 0.0, "precision": 0.0, "peak_wps": 99.0,
         "ok": False, "score": 0.0}
  if n == 0 or not words:
    return bad
  fr = x[: n * fl].reshape(n, fl)
  db = 20 * np.log10(np.sqrt((fr ** 2).mean(axis=1)) + 1e-9)
  thr = max(-50.0, float(np.percentile(db, 90)) - 28.0)
  active = db > thr
  tot = int(active.sum())
  covered = np.zeros(n, bool)
  for w in words:
    a = max(0, int((w["start"] - 0.3) / 0.05))
    b = min(n, int((w["end"] + 0.3) / 0.05) + 1)
    covered[a:b] = True
  coverage = float((active & covered).sum()) / max(1, tot)
  hit = 0
  for w in words:
    a = min(n - 1, max(0, int(w["start"] / 0.05)))
    b = min(n, max(a + 1, int(w["end"] / 0.05) + 1))
    hit += bool(active[a:b].any())
  precision = hit / len(words)
  starts = np.array([w["start"] for w in words])
  peak = 0.0
  for t in np.arange(0.0, max(float(starts.max()), 0.1), 0.5):
    peak = max(peak, float(((starts >= t) & (starts < t + 3.0)).sum()) / 3.0)
  ok = coverage >= 0.85 and precision >= 0.85 and peak <= 8.5
  return {"coverage": coverage, "precision": precision, "peak_wps": peak,
          "ok": ok, "score": coverage * precision * (1.0 if peak <= 8.5 else 0.5)}


def print_quality(name, q):
  flag = "✅" if q["ok"] else "❌"
  print(
      f"   {flag} [{name}] awaaz cover: {q['coverage']:.0%} | words awaaz par:"
      f" {q['precision']:.0%} | max speed: {q['peak_wps']:.1f} lafz/sec"
  )


def report_gaps(words, x, sr, min_gap=1.5):
  total = len(x) / sr
  prev, found = 0.0, False
  bounds = [(w["start"], w["end"]) for w in words] + [(total, total)]
  for st, en in bounds:
    if st - prev >= min_gap:
      seg = x[int(prev * sr):int(st * sr)]
      fl = int(0.05 * sr)
      n = max(1, len(seg) // fl)
      db = [20 * np.log10(np.sqrt(np.mean(seg[i*fl:(i+1)*fl] ** 2) + 1e-12))
            for i in range(n)]
      loud = float(np.mean(np.array(db) > -35))
      tag = "🔴 awaaz hai, par koi lafz nahi" if loud > 0.3 else "🟢 khamoshi"
      print(f"   GAP {prev:6.2f}s -> {st:6.2f}s ({st-prev:5.1f}s): {tag}")
      found = True
    prev = max(prev, en)
  if not found:
    print("   ✅ Koi bada gap nahi.")


# ==============================================================================
# 5. TRANSCRIPTION / ALIGNMENT METHODS
# ==============================================================================
def transcribe_whisperx(audio):
  model = whisperx.load_model(
      get_whisper_model_path(), device, compute_type=COMPUTE_TYPE,
      language=AUDIO_LANGUAGE,
      asr_options={"initial_prompt": WHISPER_INITIAL_PROMPT},
      vad_options={"vad_onset": 0.30, "vad_offset": 0.20},
      download_root=WHISPER_DIR,
  )
  result = model.transcribe(audio, batch_size=16, language=AUDIO_LANGUAGE)
  del model
  gc.collect()
  torch.cuda.empty_cache()
  return result


def align_wav2vec(result, audio, audio_dur):
  model_a, metadata = whisperx.load_align_model(
      language_code=AUDIO_LANGUAGE, device=device, model_dir=WAV2VEC_DIR)
  vocab_lower = {k.lower() for k in metadata["dictionary"].keys()}

  segs, unknown = [], {}
  for s in result["segments"]:
    for ch in s["text"]:
      if ch.strip() and ch.lower() not in vocab_lower and ch.isalpha():
        unknown[ch] = unknown.get(ch, 0) + 1
    toks = [norm_token(t, vocab_lower) for t in s["text"].split()]
    toks = [t for t in toks if t]
    if toks:
      segs.append({"text": " ".join(toks), "start": s["start"], "end": s["end"]})
  if unknown:
    top = sorted(unknown.items(), key=lambda x: -x[1])[:12]
    print(f"   ℹ️ Aligner vocab mein na milne wale chars (wildcard): {top}")
  if not segs:
    raise RuntimeError("❌ Whisper ne koi text nahi diya.")

  aligned = whisperx.align(
      segs, model_a, metadata, audio, device, return_char_alignments=False)
  del model_a
  gc.collect()
  torch.cuda.empty_cache()
  return build_words(aligned["segments"], audio_dur)


def transcribe_native(audio, audio_dur):
  """Whisper ke apne word timestamps (cross-attention) - Urdu+English mix mein
  bhi kaam karte hain, alag aligner ki zaroorat nahi."""
  from faster_whisper import WhisperModel

  fw = WhisperModel(
      get_whisper_model_path(), device=device, compute_type=COMPUTE_TYPE,
      download_root=WHISPER_DIR)
  segs, _ = fw.transcribe(
      audio, language=AUDIO_LANGUAGE, task="transcribe", beam_size=5,
      word_timestamps=True, initial_prompt=WHISPER_INITIAL_PROMPT,
      condition_on_previous_text=False, vad_filter=True,
      vad_parameters={"threshold": 0.30, "min_silence_duration_ms": 400,
                      "speech_pad_ms": 200},
  )
  aligned = []
  for s in segs:
    ws = [{"word": w.word.strip(), "start": w.start, "end": w.end}
          for w in (s.words or []) if w.word.strip()]
    if ws:
      aligned.append({"start": s.start, "end": s.end, "text": s.text,
                      "words": ws})
  del fw
  gc.collect()
  torch.cuda.empty_cache()
  if not aligned:
    raise RuntimeError("❌ Whisper native ne koi lafz nahi diya.")
  return build_words(aligned, audio_dur)


# ==============================================================================
# 6. MAIN PIPELINE
# ==============================================================================
def step1_clean_and_transcribe(
    raw_audio_path: str,
    clean_audio_path: str = OUTPUT_CLEAN_WAV,
    words_json_path: str = OUTPUT_WORDS_JSON,
    force_rerun: bool = FORCE_RERUN,
):
  drive_clean = os.path.join(PROJECT_DIR, clean_audio_path)
  drive_words = os.path.join(PROJECT_DIR, words_json_path)
  meta_file = os.path.join(PROJECT_DIR, ".last_audio_meta.json")

  cur_meta = {
      "name": os.path.basename(raw_audio_path),
      "size": os.path.getsize(raw_audio_path),
      "ver": CACHE_VERSION,
  }
  same_audio = False
  if os.path.exists(meta_file):
    try:
      with open(meta_file, "r", encoding="utf-8") as f:
        same_audio = json.load(f) == cur_meta
    except Exception:
      same_audio = False

  if (not force_rerun and same_audio and os.path.exists(drive_clean)
      and os.path.exists(drive_words)):
    shutil.copy(drive_clean, clean_audio_path)
    shutil.copy(drive_words, words_json_path)
    with open(words_json_path, "r", encoding="utf-8") as f:
      words_list = json.load(f)
    print(f"⚡ Cache hit: {len(words_list)} words loaded.")
    return words_list

  for stale in [clean_audio_path, words_json_path, RAW_MASTER_WAV,
                "final_flawless_voice.wav", "synced_final_words.json",
                "reel_visual_plan.json"]:
    if os.path.exists(stale):
      os.remove(stale)

  print(f"🎙️ Processing: {raw_audio_path}")
  master = extract_master_wav(raw_audio_path, RAW_MASTER_WAV)
  dwav, sr = load_audio_tensor_clean(master)
  audio_dur = len(dwav) / sr
  print(f"   ⏱️ Master audio duration: {audio_dur:.3f}s")

  print("1. Resemble Enhance se Aawaz Clean ho rahi hai...")
  hwav, new_sr = enhance(
      dwav, sr, device, nfe=16, solver="midpoint", lambd=0.9, tau=0.5,
      run_dir=get_resemble_run_dir(),
  )
  hwav = hwav.detach().cpu().numpy()
  expected = int(round(audio_dur * new_sr))
  if len(hwav) != expected:
    print(f"   ⚠️ Length fix: {len(hwav)} -> {expected} samples")
    hwav = (hwav[:expected] if len(hwav) > expected
            else np.pad(hwav, (0, expected - len(hwav))))
  sf.write(clean_audio_path, hwav, new_sr)
  del dwav, hwav
  gc.collect()
  torch.cuda.empty_cache()

  asr_path = master if ALIGN_ON_RAW else clean_audio_path
  audio = whisperx.load_audio(asr_path)  # 16k float32
  x_np, x_sr = load_audio_tensor_clean(asr_path)
  x_np = x_np.numpy()

  candidates = []  # (name, words, quality)

  if ALIGN_METHOD in ("auto", "wav2vec"):
    print("2. WhisperX transcribe...")
    result = transcribe_whisperx(audio)
    with open("raw_segments_debug.json", "w", encoding="utf-8") as f:
      json.dump(result["segments"], f, ensure_ascii=False, indent=2)
    print("3. Wav2Vec2 alignment...")
    words_a = align_wav2vec(result, audio, audio_dur)
    qa = alignment_quality(words_a, x_np, x_sr)
    print_quality("wav2vec2", qa)
    candidates.append(("wav2vec2", words_a, qa))

  need_native = (
      ALIGN_METHOD == "whisper_native"
      or (ALIGN_METHOD == "auto" and not candidates[0][2]["ok"])
  )
  if need_native:
    if candidates:
      print("⚠️ Wav2Vec2 timestamps awaaz se match nahi kiye -> Whisper native"
            " word timestamps try ho rahe hain...")
    else:
      print("2. Whisper native word timestamps...")
    words_b = transcribe_native(audio, audio_dur)
    qb = alignment_quality(words_b, x_np, x_sr)
    print_quality("whisper_native", qb)
    candidates.append(("whisper_native", words_b, qb))

  name, words_list, q = max(candidates, key=lambda c: c[2]["score"])
  print(f"\n🏆 Selected method: {name}")

  with open(words_json_path, "w", encoding="utf-8") as f:
    json.dump(words_list, f, ensure_ascii=False, indent=2)
  shutil.copy(clean_audio_path, drive_clean)
  shutil.copy(words_json_path, drive_words)
  with open(meta_file, "w", encoding="utf-8") as f:
    json.dump(cur_meta, f)

  est = sum(1 for w in words_list if w["est"])
  long_w = [w for w in words_list if (w["end"] - w["start"]) > 1.5]
  print("\n✅ STEP 2 COMPLETE")
  print(f"   🔊 Clean audio : {clean_audio_path}")
  print(f"   📝 Words       : {len(words_list)} -> {words_json_path}")
  print(f"   📊 Aligned     : {len(words_list) - est} | Estimated: {est}")
  print(f"   ⏱️ First word {words_list[0]['start']}s, last ends"
        f" {words_list[-1]['end']}s (audio {audio_dur:.2f}s)")
  print(f"   📂 Source file : {raw_audio_path}")
  print("   🔎 Gap check:")
  report_gaps(words_list, x_np, x_sr)
  if long_w:
    print(f"   ⚠️ {len(long_w)} words 1.5s se lambe: "
          f"{[(w['word'], w['start']) for w in long_w[:8]]}")
  if not q["ok"]:
    print("   ❌ Warning: final timestamps bhi 100% match nahi. Step 3 mein"
          " words ka sync dhyan se check karein.")
  return words_list


words = step1_clean_and_transcribe(
    raw_audio_path=RAW_AUDIO_PATH,
    clean_audio_path=OUTPUT_CLEAN_WAV,
    words_json_path=OUTPUT_WORDS_JSON,
    force_rerun=FORCE_RERUN,
)

In [ ]:
print(words)

# ✂️ Step 3: Interactive CapCut-Style Waveform Editor & Auto-Sync Engine

This cell launches a browser-based **CapCut-Style Interactive Audio Timeline Editor** directly inside Google Colab. It allows you to visually inspect the audio waveform alongside live synchronized Urdu/English words, trim out mistakes or awkward pauses, and export a studio-mastered final track with automatically recalculated timestamps.

### ⚡ Key Features & Workflow:
* **Real-Time Waveform & Live Word Preview:** Renders a smooth, zoomable waveform (`WaveSurfer.js`) that highlights each spoken word in real time as the playhead moves.
* **Non-Destructive Split & Cut Controls:** Use keyboard shortcuts (`Space` to Play/Pause, `S` to Split at Playhead, `D` to Cut/Restore a clip) to mark unwanted segments in red. The preview player automatically skips cut regions during playback.
* **Lossless Stitching & Crossfading:** When you click **Export**, Python slices the original lossless `step1_clean.wav`, stitches kept intervals together with a seamless `15ms` crossfade, and applies broadcast loudness normalization (`-14 LUFS`).
* **Automatic Timestamp & Transcript Recalibration:** Recalculates exact millisecond boundaries for all remaining words (`synced_final_words.json`), generates a clean text transcript (`synced_final_transcript.txt`), and backs up all outputs permanently to Google Drive.

In [ ]:
# ==============================================================================
# ✂️ STEP 3 (v4): CAPCUT-STYLE WAVEFORM EDITOR & AUTO-SYNC ENGINE
# ==============================================================================
import base64
import json
import os
import shutil
import subprocess
import sys
import numpy as np
from google.colab import output
from IPython.display import Audio, HTML, display
from pydub import AudioSegment

# ==============================================================================
# 🎯 0. USER SETTINGS
# ==============================================================================
PROJECT_FOLDER_MAIN = "/content/drive/MyDrive/Video Editing"
PROJECT_FOLDER_FALLBACK = "/content/drive/MyDrive/Colab Notebooks/Video Editing"
DRIVE_ENV_DIR = "/content/drive/MyDrive/Colab_AI_Env"

INPUT_CLEAN_AUDIO_NAME = "step1_clean.wav"
INPUT_RAW_WORDS_JSON = "raw_words.json"

OUTPUT_FINAL_AUDIO_WAV = "final_flawless_voice.wav"
OUTPUT_SYNCED_WORDS_JSON = "synced_final_words.json"
OUTPUT_TRANSCRIPT_TXT = "synced_final_transcript.txt"

CROSSFADE_MS = 15
MIN_KEEP_CLIP_SEC = 0.05
PREVIEW_MP3_BITRATE = "96k"
PREVIEW_WAV_MAX_SEC = 240  # is se chhoti audio ka preview WAV (exact timing), warna MP3
DEFAULT_ZOOM_LEVEL = 45
FFMPEG_MASTERING_FILTER = "highpass=f=80,equalizer=f=3200:width_type=h:width=200:g=2,loudnorm=I=-14:TP=-1.5:LRA=11"
MASTER_SAMPLE_RATE = "48000"

# ==============================================================================
# 1. PATHS
# ==============================================================================
if DRIVE_ENV_DIR not in sys.path and os.path.exists(DRIVE_ENV_DIR):
  sys.path.insert(0, DRIVE_ENV_DIR)

PROJECT_DIR = (
    PROJECT_FOLDER_MAIN
    if os.path.exists(PROJECT_FOLDER_MAIN)
    else PROJECT_FOLDER_FALLBACK
)
os.makedirs(PROJECT_DIR, exist_ok=True)


def resolve_input_file(filename: str) -> str:
  drive_path = os.path.join(PROJECT_DIR, filename)
  if os.path.exists(drive_path):
    if not os.path.exists(filename):
      shutil.copy(drive_path, filename)
    return drive_path
  elif os.path.exists(filename):
    shutil.copy(filename, drive_path)
    return filename
  else:
    raise FileNotFoundError(
        f"❌ Input file nahi mili: '{filename}'. Pehle Step 2 chalayein!"
    )


def audio_to_np(seg: AudioSegment):
  seg = seg.set_channels(1)
  x = np.array(seg.get_array_of_samples(), dtype=np.float32)
  return x / float(1 << (8 * seg.sample_width - 1)), seg.frame_rate


def quick_sync_check(words, seg: AudioSegment):
  """Step 2 ke words audio se match karte hain ya nahi - editor khulne se pehle."""
  x, sr = audio_to_np(seg)
  dur = len(x) / sr
  fl = int(0.05 * sr)
  n = len(x) // fl
  if n == 0 or not words:
    print("⚠️ Words ya audio khali hai!")
    return
  fr = x[: n * fl].reshape(n, fl)
  db = 20 * np.log10(np.sqrt((fr ** 2).mean(axis=1)) + 1e-9)
  thr = max(-50.0, float(np.percentile(db, 90)) - 28.0)
  active = db > thr
  covered = np.zeros(n, bool)
  for w in words:
    a = max(0, int((w["start"] - 0.3) / 0.05))
    b = min(n, int((w["end"] + 0.3) / 0.05) + 1)
    covered[a:b] = True
  cov = float((active & covered).sum()) / max(1, int(active.sum()))
  last_end = max(w["end"] for w in words)
  print(f"🔎 Sync check: audio {dur:.1f}s | last word ends {last_end:.1f}s |"
        f" awaaz cover {cov:.0%}")
  if last_end > dur + 0.5:
    print("   ❌ Words audio se lambe hain! Ye words kisi aur audio ke hain.")
  if cov < 0.85:
    print("   ❌ Words audio se match NAHI karte (editor mein lafz aage/peeche"
          " dikhenge). Step 2 dobara chalayein (v4) - ye JSON purani/galat hai.")
  else:
    print("   ✅ Words audio ke saath sync hain.")


# ==============================================================================
# 2. EDITOR
# ==============================================================================
def launch_capcut_waveform_editor(
    clean_audio_name: str = INPUT_CLEAN_AUDIO_NAME,
    words_json_name: str = INPUT_RAW_WORDS_JSON,
    final_output: str = OUTPUT_FINAL_AUDIO_WAV,
    final_json_output: str = OUTPUT_SYNCED_WORDS_JSON,
    final_txt_output: str = OUTPUT_TRANSCRIPT_TXT,
):
  clean_audio_path = resolve_input_file(clean_audio_name)
  words_json_path = resolve_input_file(words_json_name)

  print(f"🎙️ Audio : {clean_audio_path}")
  print(f"📝 Words : {words_json_path}")

  with open(words_json_path, "r", encoding="utf-8") as f:
    words_list = json.load(f)

  src_seg = AudioSegment.from_file(clean_audio_path)
  quick_sync_check(words_list, src_seg)

  # Preview: chhoti audio ke liye 16k WAV (mp3 ka encoder-delay shift nahi hota)
  if len(src_seg) / 1000.0 <= PREVIEW_WAV_MAX_SEC:
    preview_file, preview_mime = "temp_waveform_preview.wav", "audio/wav"
    cmd = ["ffmpeg", "-y", "-i", clean_audio_path, "-ac", "1", "-ar", "16000",
           "-c:a", "pcm_s16le", preview_file]
  else:
    preview_file, preview_mime = "temp_waveform_preview.mp3", "audio/mpeg"
    cmd = ["ffmpeg", "-y", "-i", clean_audio_path, "-b:a",
           PREVIEW_MP3_BITRATE, "-ac", "1", preview_file]
  subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                 check=True)
  with open(preview_file, "rb") as f:
    b64_audio = base64.b64encode(f.read()).decode("utf-8")

  def _save_cuts_from_js(deleted_ranges_json):
    deleted_ranges = json.loads(deleted_ranges_json)
    print(f"\n⏳ Applying {len(deleted_ranges)} cuts to lossless"
          f" '{clean_audio_path}'...")

    source_audio = AudioSegment.from_file(clean_audio_path)
    total_dur_sec = len(source_audio) / 1000.0

    deleted_ranges.sort(key=lambda x: x["start"])
    merged_del = []
    for r in deleted_ranges:
      if not merged_del or r["start"] > merged_del[-1]["end"]:
        merged_del.append({"start": r["start"], "end": r["end"]})
      else:
        merged_del[-1]["end"] = max(merged_del[-1]["end"], r["end"])

    kept_intervals = []
    curr = 0.0
    for r in merged_del:
      if r["start"] > curr + MIN_KEEP_CLIP_SEC:
        kept_intervals.append((curr, r["start"]))
      curr = max(curr, r["end"])
    if curr < total_dur_sec - MIN_KEEP_CLIP_SEC:
      kept_intervals.append((curr, total_dur_sec))

    if not kept_intervals:
      print("❌ Saari audio cut ho gayi - kuch export nahi hua.")
      return

    final_audio = AudioSegment.empty()
    updated_timeline = []
    current_out_ms = 0  # final_audio ki current length (ms)

    for s_sec, e_sec in kept_intervals:
      s_ms = int(round(s_sec * 1000))
      e_ms = int(round(e_sec * 1000))
      chunk = source_audio[s_ms:e_ms]

      # Crossfade ke baad naya chunk CROSSFADE_MS pehle se shuru hota hai
      first = len(final_audio) == 0
      chunk_start_out_ms = current_out_ms - (0 if first else CROSSFADE_MS)
      offset_sec = (chunk_start_out_ms - s_ms) / 1000.0

      for w in words_list:
        w_mid = (w["start"] + w["end"]) / 2.0
        if s_sec <= w_mid <= e_sec:
          updated_timeline.append({
              "id": len(updated_timeline),
              "word": w["word"],
              "start": round(max(0.0, w["start"] + offset_sec), 3),
              "end": round(max(0.0, w["end"] + offset_sec), 3),
          })

      if first:
        final_audio = chunk
        current_out_ms = len(chunk)
      else:
        final_audio = final_audio.append(chunk, crossfade=CROSSFADE_MS)
        current_out_ms = len(final_audio)

    temp_out = "temp_capcut_stitched.wav"
    final_audio.export(temp_out, format="wav")

    subprocess.run(
        ["ffmpeg", "-y", "-i", temp_out, "-af", FFMPEG_MASTERING_FILTER,
         "-ar", MASTER_SAMPLE_RATE, final_output],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

    with open(final_json_output, "w", encoding="utf-8") as f:
      json.dump(updated_timeline, f, ensure_ascii=False, indent=2)
    with open(final_txt_output, "w", encoding="utf-8") as f:
      f.write(" ".join(w["word"] for w in updated_timeline) + "\n")

    drive_wav = os.path.join(PROJECT_DIR, final_output)
    drive_json = os.path.join(PROJECT_DIR, final_json_output)
    drive_txt = os.path.join(PROJECT_DIR, final_txt_output)
    shutil.copy(final_output, drive_wav)
    shutil.copy(final_json_output, drive_json)
    shutil.copy(final_txt_output, drive_txt)

    print(f"✅ DONE! Final Audio : {final_output}"
          f" ({len(final_audio)/1000.0:.1f}s) -> Drive: {drive_wav}")
    print(f"✅ Synced Words      : {final_json_output}"
          f" ({len(updated_timeline)} words) -> Drive: {drive_json}")
    print(f"📄 Transcript (.txt) : {final_txt_output} -> Drive: {drive_txt}")
    display(Audio(final_output))

  output.register_callback("notebook.save_capcut_edits", _save_cuts_from_js)

  html_code = f"""
  <div id="capcut-app" style="background:#121316; color:#f1f1f1; padding:18px; border-radius:12px; font-family:system-ui,sans-serif; border:1px solid #2a2d35;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:12px; flex-wrap:wrap; gap:10px;">
      <div>
        <span style="font-size:17px; font-weight:700; color:#00e5ff;">✂️ CapCut-Style Waveform Timeline Editor</span>
        <span id="time-display" style="margin-left:15px; font-family:monospace; background:#1e2028; padding:4px 10px; border-radius:6px; color:#00e676; font-size:15px;">00:00.00 / 00:00.00</span>
      </div>
      <div style="display:flex; align-items:center; gap:8px;">
        <label style="font-size:13px; color:#aaa;">🔍 Zoom:</label>
        <input id="zoom-slider" type="range" min="15" max="250" value="{DEFAULT_ZOOM_LEVEL}" style="width:130px; accent-color:#00e5ff;">
      </div>
    </div>

    <div id="word-preview" style="background:#1a1c23; padding:8px 14px; border-radius:8px; margin-bottom:12px; min-height:28px; font-size:16px; color:#ffd54f; text-align:right; direction:rtl; border-left:4px solid #00e5ff;">
      آڈیو لوڈ ہو رہی ہے... (Loading Waveform...)
    </div>

    <div id="waveform" style="background:#181a20; border-radius:8px; border:1px solid #2e323d; padding:10px 0; overflow-x:auto;"></div>

    <div style="display:flex; justify-content:space-between; align-items:center; margin-top:14px; flex-wrap:wrap; gap:10px;">
      <div style="display:flex; gap:8px; flex-wrap:wrap;">
        <button id="btn-play" style="background:#00e5ff; color:#000; border:none; padding:9px 18px; border-radius:7px; font-weight:700; cursor:pointer;">▶ Play / Pause (Space)</button>
        <button id="btn-split" style="background:#2979ff; color:#fff; border:none; padding:9px 16px; border-radius:7px; font-weight:600; cursor:pointer;">✂️ Split at Playhead (S)</button>
        <button id="btn-del" style="background:#ff1744; color:#fff; border:none; padding:9px 16px; border-radius:7px; font-weight:600; cursor:pointer;">🗑 Cut / Restore Current Clip (D)</button>
        <button id="btn-undo" style="background:#37474f; color:#fff; border:none; padding:9px 14px; border-radius:7px; cursor:pointer;">↩ Clear All Splits</button>
      </div>
      <button id="btn-export" style="background:#00c853; color:#fff; border:none; padding:10px 22px; border-radius:7px; font-weight:700; cursor:pointer; font-size:14px;">🎬 Export Final Cut Audio + .TXT</button>
    </div>

    <div style="margin-top:12px; font-size:12px; color:#9e9e9e;">
      💡 <b>Use:</b> Play -> <b>S</b> se split (galti se pehle aur baad) -> galti wale clip par <b>D</b> dabayein, wo <span style="color:#ff5252;">RED (Cut)</span> ho jayega aur player use skip karega. Kisi lafz par click karein to wahan seek hoga (sync check).
    </div>
    <div id="segments-bar" style="display:flex; gap:6px; flex-wrap:wrap; margin-top:10px; max-height:110px; overflow-y:auto; padding:6px; background:#181a20; border-radius:6px;"></div>
    <div id="export-status" style="margin-top:10px; color:#00e676; font-weight:600;"></div>
  </div>

  <script type="module">
    import WaveSurfer from 'https://unpkg.com/wavesurfer.js@7/dist/wavesurfer.esm.js';
    import RegionsPlugin from 'https://unpkg.com/wavesurfer.js@7/dist/plugins/regions.esm.js';
    import TimelinePlugin from 'https://unpkg.com/wavesurfer.js@7/dist/plugins/timeline.esm.js';

    const wordsData = {json.dumps(words_list)};
    const b64Audio = "{b64_audio}";

    const byteCharacters = atob(b64Audio);
    const byteNumbers = new Array(byteCharacters.length);
    for (let i = 0; i < byteCharacters.length; i++) byteNumbers[i] = byteCharacters.charCodeAt(i);
    const audioBlob = new Blob([new Uint8Array(byteNumbers)], {{ type: '{preview_mime}' }});
    const audioUrl = URL.createObjectURL(audioBlob);

    const wsRegions = RegionsPlugin.create();
    const ws = WaveSurfer.create({{
      container: '#waveform',
      waveColor: '#4fc3f7',
      progressColor: '#00e5ff',
      cursorColor: '#ffeb3b',
      cursorWidth: 2,
      height: 115,
      minPxPerSec: {DEFAULT_ZOOM_LEVEL},
      normalize: true,
      url: audioUrl,
      plugins: [wsRegions, TimelinePlugin.create({{ height: 22 }})]
    }});

    let splitPoints = [0];
    let deletedSegments = new Set();
    let duration = 0;

    function fmtTime(s) {{
      const mins = Math.floor(s / 60).toString().padStart(2, '0');
      const secs = (s % 60).toFixed(2).padStart(5, '0');
      return `${{mins}}:${{secs}}`;
    }}

    function renderSegmentsUI() {{
      wsRegions.clearRegions();
      const bar = document.getElementById('segments-bar');
      bar.innerHTML = '';
      splitPoints.sort((a, b) => a - b);

      for (let i = 0; i < splitPoints.length - 1; i++) {{
        const s = splitPoints[i];
        const e = splitPoints[i + 1];
        const key = s.toFixed(2);
        const isCut = deletedSegments.has(key);

        if (isCut) {{
          wsRegions.addRegion({{
            start: s, end: e, color: 'rgba(255, 23, 68, 0.48)', drag: false, resize: false
          }});
        }} else if (i > 0) {{
          wsRegions.addRegion({{
            start: s, end: s + 0.03, color: '#ffeb3b', drag: false, resize: false
          }});
        }}

        const chip = document.createElement('div');
        chip.style.cssText = `padding:5px 10px; border-radius:5px; cursor:pointer; font-size:12px; font-family:monospace; user-select:none; border:1px solid ${{isCut ? '#ff1744' : '#2e323d'}}; background:${{isCut ? '#3d0e18' : '#22252e'}}; color:${{isCut ? '#ff8a80' : '#b0bec5'}};`;
        chip.innerHTML = `Clip ${{i+1}} [${{s.toFixed(1)}}s-${{e.toFixed(1)}}s] ${{isCut ? '❌ CUT' : '✅ KEEP'}}`;
        chip.onclick = () => {{ ws.setTime(s + 0.02); }};
        chip.ondblclick = () => {{
          if (deletedSegments.has(key)) deletedSegments.delete(key);
          else deletedSegments.add(key);
          renderSegmentsUI();
        }};
        bar.appendChild(chip);
      }}
    }}

    // Har frame par chalta hai (audioprocess) - cut skip aur word highlight dono accurate
    function tick(currentTime) {{
      document.getElementById('time-display').innerText = `${{fmtTime(currentTime)}} / ${{fmtTime(duration)}}`;

      if (ws.isPlaying()) {{
        for (let i = 0; i < splitPoints.length - 1; i++) {{
          const s = splitPoints[i];
          const e = splitPoints[i + 1];
          if (currentTime >= s && currentTime < e - 0.005 && deletedSegments.has(s.toFixed(2))) {{
            if (e >= duration - 0.05) {{
              ws.pause();
              ws.setTime(Math.max(0, s - 0.01));
            }} else {{
              ws.setTime(e);
            }}
            return;
          }}
        }}
      }}

      const nearby = wordsData.filter(w => w.end >= currentTime - 1.8 && w.start <= currentTime + 1.8);
      if (nearby.length > 0) {{
        const started = nearby.filter(w => w.start <= currentTime);
        const cur = started.length ? started[started.length - 1] : null;
        const curId = (cur && currentTime <= cur.end + 0.25) ? cur.id : -1;
        document.getElementById('word-preview').innerHTML = nearby.map(w => {{
          const active = (w.id === curId);
          return `<span data-t="${{w.start}}" style="cursor:pointer; padding:2px 5px; border-radius:4px; background:${{active ? '#00e5ff' : 'transparent'}}; color:${{active ? '#000' : '#ffd54f'}}; font-weight:${{active ? '700' : '400'}};">${{w.word}}</span>`;
        }}).join(' ');
      }}
    }}

    ws.on('ready', () => {{
      duration = ws.getDuration();
      splitPoints = [0, duration];
      document.getElementById('word-preview').innerText = 'تیار ہے! Play دبائیں، جہاں غلطی ہو وہاں S دبا کر Split کریں اور D دبا کر Delete کریں۔';
      document.getElementById('time-display').innerText = `00:00.00 / ${{fmtTime(duration)}}`;
      renderSegmentsUI();
    }});

    ws.on('audioprocess', tick);
    ws.on('timeupdate', tick);

    document.getElementById('word-preview').onclick = (e) => {{
      const t = e.target && e.target.dataset ? e.target.dataset.t : null;
      if (t !== undefined && t !== null && t !== '') ws.setTime(parseFloat(t));
    }};

    document.getElementById('zoom-slider').oninput = (e) => {{
      ws.zoom(Number(e.target.value));
    }};

    document.getElementById('btn-play').onclick = () => ws.playPause();

    function splitAtPlayhead() {{
      const t = ws.getCurrentTime();
      if (t <= 0.05 || t >= duration - 0.05) return;
      if (!splitPoints.some(p => Math.abs(p - t) < 0.03)) {{
        splitPoints.push(t);
        renderSegmentsUI();
      }}
    }}
    document.getElementById('btn-split').onclick = splitAtPlayhead;

    function toggleDeleteCurrentClip() {{
      const t = ws.getCurrentTime();
      for (let i = 0; i < splitPoints.length - 1; i++) {{
        if (t >= splitPoints[i] && t <= splitPoints[i + 1]) {{
          const key = splitPoints[i].toFixed(2);
          if (deletedSegments.has(key)) deletedSegments.delete(key);
          else deletedSegments.add(key);
          renderSegmentsUI();
          break;
        }}
      }}
    }}
    document.getElementById('btn-del').onclick = toggleDeleteCurrentClip;

    document.getElementById('btn-undo').onclick = () => {{
      splitPoints = [0, duration];
      deletedSegments.clear();
      renderSegmentsUI();
    }};

    document.addEventListener('keydown', (e) => {{
      if (['INPUT', 'TEXTAREA'].includes(e.target.tagName)) return;
      if (e.code === 'Space') {{ e.preventDefault(); ws.playPause(); }}
      else if (e.key.toLowerCase() === 's') {{ e.preventDefault(); splitAtPlayhead(); }}
      else if (e.key.toLowerCase() === 'd') {{ e.preventDefault(); toggleDeleteCurrentClip(); }}
    }});

    document.getElementById('btn-export').onclick = async () => {{
      const ranges = [];
      for (let i = 0; i < splitPoints.length - 1; i++) {{
        const key = splitPoints[i].toFixed(2);
        if (deletedSegments.has(key)) {{
          ranges.push({{ start: splitPoints[i], end: splitPoints[i + 1] }});
        }}
      }}
      document.getElementById('export-status').innerText = '⏳ Exporting audio, JSON & .TXT transcript to Drive... (Check below)';
      await google.colab.kernel.invokeFunction('notebook.save_capcut_edits', [JSON.stringify(ranges)], {{}});
      document.getElementById('export-status').innerText = '✅ Export Complete! Saved {final_output}, {final_json_output} & {final_txt_output} (Backed up to Drive)';
    }};
  </script>
  """
  display(HTML(html_code))


launch_capcut_waveform_editor()

# 🎬 Step 4: AI Visual Storyboard Director, SFX Mixer & Pro Split-Screen Compositor

This is the master rendering engine of the **Universal AI Reel Generator**. It takes the synchronized word timestamps and clean audio from the previous steps, directs a complete visual and typographic storyboard using **Qwen2.5-7B**, generates photorealistic scene assets via **LCM-DreamShaper**, synthesizes custom transition sound effects, and composites a broadcast-ready **1080x1920 vertical video**.

### ⚡ Key Pipeline Capabilities:
* **Intelligent Storyboard & Takeaway Direction (`Qwen2.5-7B`):** Analyzes the transcript to automatically generate scene-specific photorealistic image prompts, punchy 3–5 word headlines, semantic category tags (`TIP`, `MISTAKE`, `FIX`, `FACT`), dynamic 3D emoji stickers, and a 3-point end-of-video summary (`REMEMBER` recap card).
* **Automatic Bilingual Loanword Transliteration:** Detects English technical terms written in Urdu script (e.g., `کیش` $\rightarrow$ `CACHE`, `ڈیٹا` $\rightarrow$ `DATA`) and renders them in crisp uppercase Latin typography alongside HarfBuzz/Raqm-shaped RTL Urdu text.
* **Photorealistic Scene Generation (`LCM-DreamShaper`):** Synthesizes vibrant, high-contrast 8K-style visuals for every 3.5-second segment, with smart caching so existing scenes are reused instantaneously.
* **Procedural Sound Effects (SFX) Mastering:** Synthesizes and mixes custom *whoosh* transition sweeps, *pop* badge cues, and *ding* recap chimes directly into the master audio track.
* **Pro Split-Screen Motion Graphics (1080x1920 @ 25 FPS):**
  * **Top Canvas:** Smooth 4-way Ken Burns camera motion (zoom-in, zoom-out, pan-left, pan-right), scene-cut flash transitions, animated spring-physics emoji stickers, and a timed *"SAVE THIS FOR LATER"* retention nudge.
  * **Bottom Cyber Card:** Color-coded semantic tags, auto-fitted English headlines, a voice-reactive pulsing presenter avatar (`AHSAN TARIQ`), and millisecond-accurate RTL Urdu/English karaoke word highlighting.

In [ ]:
# ==============================================================================
# 🎬 STEP 4: PRO EDITORIAL REEL DIRECTOR, MULTI-SFX ENGINE & COMPOSITOR
# (Zero Installs | 100% Copy-Paste Safe | Anti-AI Matte Editorial Themes)
# ==============================================================================
sub = operator.sub
mul = operator.mul

# ==============================================================================
# 🎯 0. MASTER CONTROL PANEL (Har Cheez Yahan Se Customize Karein!)
# ==============================================================================

# 👉 [A. INPUT & OUTPUT FILES (Auto-connected with Drive & Step 2/3)]:
INPUT_VOICE_WAV        = "final_flawless_voice.wav"     # Step 3 output (auto-falls back to step1_clean.wav)
INPUT_WORDS_JSON       = "synced_final_words.json"      # Step 3 output (auto-falls back to raw_words.json)
INPUT_AVATAR_IMAGE     = "studio_presenter.png"         # Master Cell 1 studio avatar
OUTPUT_VIDEO_MP4       = "Ahsan_Editorial_Reel_Pro.mp4" # Final MP4 filename (Saved to Local + Drive)
OUTPUT_MIXED_AUDIO_WAV = "master_audio_with_sfx.wav"    # Audio mixed with multi-SFX
OUTPUT_PLAN_JSON       = "reel_visual_plasn.json"        # Cached Qwen visual plan
OUTPUT_RECAP_JSON      = "reel_recap.json"              # Cached 3 takeaways
OUTPUT_TOPIC_JSON      = "reel_topic.json"              # Cached video title
OUTPUT_EN_MAP_JSON     = "reel_en_map.json"             # Cached Urdu->English loanword map
OUTPUT_SCENE_IMG_DIR   = "reel_scene_images"            # Folder for generated scene images

# 👉 [B. GENERATION & TIMING CONTROLS]:
FORCE_REGEN            = True  # True = Qwen plan + images dobara banao | False = Drive/Local cache use karo
VIDEO_FPS              = 25     # Frame rate
TARGET_SCENE_SEC       = 3.5    # Har scene ki average duration (seconds)
MAX_WORDS_PER_SCENE    = 12     # Ek scene mein max words
RECAP_DUR              = 4.5    # End par "KEY TAKEAWAYS" card kitne seconds dikhe
SAVE_NUDGE_AT          = 0.60   # Video ke kitne % hisse par "SAVE FOR LATER" badge aaye (0.0 to 1.0)
SHOW_EMOJI_STICKER     = True   # Top-right animated emoji dikhana hai ya nahi

# 👉 [C. FIXED TOP TITLE & BRANDING TEXTS]:
SHOW_TITLE             = True   # True = Poori video mein upar topic title FIXED dikhao
TITLE_OVERRIDE         = ""     # Khali "" = Qwen khud topic nikalega | Ya apna likhein e.g. "HOW CACHING WORKS"
PRESENTER_NAME_TEXT    = "AHSAN TARIQ"
WATERMARK_TEXT         = "@AhsanTariq0724"
FOLLOW_CTA_TEXT        = "FOLLOW  @AhsanTariq0724"
SAVE_NUDGE_TEXT        = "SAVE THIS FOR LATER"
RECAP_HEADER_TEXT      = "KEY TAKEAWAYS"

# 👉 [D. LANGUAGE & LOANWORD SETTINGS]:
USE_NASTALIQ           = False  # False = Clean Modern Naskh/Sans | True = Nastaliq
CONVERT_LOANWORDS      = True   # Urdu haroof mein likhe English lafz (e.g. کیش) ko English mein dikhao
EN_OVERRIDES           = {"کیش": "cache", "کیشے": "cache", "ڈیٹا": "data", "ویب": "web", "سلو": "slow"}

# 👉 [E. MULTI-SOUND EFFECTS (SFX) CONTROLS (5 Different Organic Studio Sounds!)]:
ENABLE_SFX             = True   # False = Koi SFX nahi | True = Har scene par mukhtalif organic SFX
SFX_TRANSITION_VOL_DB  = -11    # Scene change sounds volume (Soft Air Swish / Deep Sub Thud / Tape Swoosh)
SFX_ACCENT_VOL_DB      = -15    # Badge pop sounds volume (Mechanical Key Tick / Wood Tap / Glass Pluck)
SFX_OUTRO_CHIME_VOL_DB = -12    # End recap card warm chord chime volume

# 👉 [F. COLOR THEME PRESET (Say Goodbye to Fake AI Blue/Cyan!)]:
# Choose one:
#   "EDITORIAL_CHARCOAL" -> Matte Carbon Black + Warm Amber Gold + Cream Text (Johnny Harris / MKBHD Vibe)
#   "WARM_ESPRESSO"      -> Deep Coffee Mocha + Terracotta Orange + Warm Ivory (Cozy Podcast / Storytelling Vibe)
#   "MINIMAL_SLATE"      -> Nordic Graphite + Emerald Mint + Crisp White (Clean Modern Tech / FinTech Vibe)
#   "ROYAL_OBSIDIAN"     -> Deep Obsidian + Champagne Gold + Soft Silver (Luxury Documentary Vibe)
THEME_PRESET = "EDITORIAL_CHARCOAL"

THEMES = {
    "EDITORIAL_CHARCOAL": {
        "bg_main":         (16, 16, 18),      # True Matte Carbon (Not AI Blue!)
        "card_bg":         (26, 26, 30),      # Elevated Matte Charcoal Card
        "card_border":     (52, 52, 60),      # Subtle metallic hairline border
        "divider_line":    (62, 62, 72),      # Clean subtle divider
        "accent_primary":  (245, 183, 59),    # Warm Editorial Amber Gold
        "accent_secondary":(255, 122, 69),    # Warm Coral Accent
        "text_on_accent":  (16, 16, 18),      # Dark ink text on gold pills
        "headline_color":  (250, 246, 238),   # Crisp Warm Cream Headline
        "urdu_spoken":     (242, 240, 234),   # Spoken words (Warm Off-White)
        "urdu_upcoming":   (125, 125, 135),   # Unspoken words (Muted Zinc)
        "title_bg":        (22, 22, 26, 235), # Fixed Top Title Box BG
        "title_border":    (245, 183, 59, 255),
        "title_text":      (250, 246, 238, 255),
        "progress_bg":     (42, 42, 48),
        "watermark_color": (165, 165, 175),
        "tag_colors": {
            "tip":     (245, 183, 59),  # Amber Gold
            "mistake": (239, 83, 80),   # Matte Crimson
            "fix":     (66, 189, 135),  # Sage Emerald
            "fact":    (224, 218, 204), # Warm Sand
        },
    },
    "WARM_ESPRESSO": {
        "bg_main":         (22, 17, 15),
        "card_bg":         (34, 27, 24),
        "card_border":     (72, 56, 48),
        "divider_line":    (68, 54, 46),
        "accent_primary":  (238, 148, 84),
        "accent_secondary":(214, 104, 83),
        "text_on_accent":  (22, 17, 15),
        "headline_color":  (252, 244, 235),
        "urdu_spoken":     (245, 236, 226),
        "urdu_upcoming":   (138, 122, 112),
        "title_bg":        (30, 23, 20, 235),
        "title_border":    (238, 148, 84, 255),
        "title_text":      (252, 244, 235, 255),
        "progress_bg":     (54, 42, 36),
        "watermark_color": (175, 160, 150),
        "tag_colors": {
            "tip":     (238, 148, 84),
            "mistake": (229, 89, 82),
            "fix":     (102, 187, 126),
            "fact":    (232, 206, 178),
        },
    },
    "MINIMAL_SLATE": {
        "bg_main":         (18, 21, 24),
        "card_bg":         (28, 33, 38),
        "card_border":     (58, 68, 78),
        "divider_line":    (55, 65, 75),
        "accent_primary":  (78, 212, 158),
        "accent_secondary":(245, 194, 78),
        "text_on_accent":  (14, 18, 20),
        "headline_color":  (244, 248, 250),
        "urdu_spoken":     (238, 242, 245),
        "urdu_upcoming":   (120, 132, 142),
        "title_bg":        (24, 28, 32, 235),
        "title_border":    (78, 212, 158, 255),
        "title_text":      (244, 248, 250, 255),
        "progress_bg":     (44, 52, 58),
        "watermark_color": (160, 172, 182),
        "tag_colors": {
            "tip":     (78, 212, 158),
            "mistake": (242, 95, 92),
            "fix":     (92, 198, 242),
            "fact":    (245, 194, 78),
        },
    },
    "ROYAL_OBSIDIAN": {
        "bg_main":         (14, 14, 17),
        "card_bg":         (24, 24, 29),
        "card_border":     (64, 58, 48),
        "divider_line":    (58, 54, 46),
        "accent_primary":  (228, 195, 132),
        "accent_secondary":(198, 162, 102),
        "text_on_accent":  (14, 14, 17),
        "headline_color":  (248, 245, 238),
        "urdu_spoken":     (240, 238, 232),
        "urdu_upcoming":   (128, 126, 122),
        "title_bg":        (20, 20, 24, 235),
        "title_border":    (228, 195, 132, 255),
        "title_text":      (248, 245, 238, 255),
        "progress_bg":     (44, 42, 38),
        "watermark_color": (168, 164, 156),
        "tag_colors": {
            "tip":     (228, 195, 132),
            "mistake": (224, 92, 88),
            "fix":     (108, 188, 148),
            "fact":    (212, 210, 204),
        },
    },
}

PAL = THEMES.get(THEME_PRESET, THEMES["EDITORIAL_CHARCOAL"])
BG_MAIN          = PAL["bg_main"]
CARD_BG          = PAL["card_bg"]
CARD_BORDER      = PAL["card_border"]
DIVIDER_COLOR    = PAL["divider_line"]
ACCENT_MAIN      = PAL["accent_primary"]
TEXT_ON_ACCENT   = PAL["text_on_accent"]
HEADLINE_COLOR   = PAL["headline_color"]
URDU_SPOKEN      = PAL["urdu_spoken"]
URDU_UPCOMING    = PAL["urdu_upcoming"]
TITLE_BG_RGBA    = PAL["title_bg"]
TITLE_BORDER_RGBA= PAL["title_border"]
TITLE_TEXT_RGBA  = PAL["title_text"]
PROGRESS_BG      = PAL["progress_bg"]
WATERMARK_COLOR  = PAL["watermark_color"]
TAG_COLORS       = PAL["tag_colors"]

# ==============================================================================
# 1. CONNECT GOOGLE DRIVE PATHS & RESOLVE INPUT FILES
# ==============================================================================
AUDIO_PATH = INPUT_VOICE_WAV if os.path.exists(INPUT_VOICE_WAV) else (
    os.path.join(PROJECT_DIR, INPUT_VOICE_WAV) if os.path.exists(os.path.join(PROJECT_DIR, INPUT_VOICE_WAV)) else (
        "step1_clean.wav" if os.path.exists("step1_clean.wav") else os.path.join(PROJECT_DIR, "step1_clean.wav")
    )
)
AVATAR_PATH = INPUT_AVATAR_IMAGE if os.path.exists(INPUT_AVATAR_IMAGE) else os.path.join(PROJECT_DIR, "ahsan_studio_avatar.png")
WORDS_JSON = INPUT_WORDS_JSON if os.path.exists(INPUT_WORDS_JSON) else os.path.join(PROJECT_DIR, INPUT_WORDS_JSON)
if not os.path.exists(WORDS_JSON):
  WORDS_JSON = "raw_words.json" if os.path.exists("raw_words.json") else os.path.join(PROJECT_DIR, "raw_words.json")

with open(WORDS_JSON, "r", encoding="utf-8") as f:
  words_data = json.load(f)

print(f"🎨 Active Theme       : {THEME_PRESET} (Matte Editorial Studio)")
print(f"🎙️ Active Audio Track : {AUDIO_PATH}")
print(f"📝 Active Words JSON  : {WORDS_JSON} ({len(words_data)} words)")
print(f"🧑 Active Avatar      : {AVATAR_PATH}")

# ==============================================================================
# 1b. FONTS & URDU SHAPING HELPERS (100% Copy-Paste Safe Math)
# ==============================================================================
W, H = 1080, 1920
TOP_H = 1040

def find_font(patterns):
  for pat in patterns:
    hits = sorted(glob.glob(f"/usr/share/fonts/**/{pat}", recursive=True))
    if hits:
      return hits[0]
  return None

EN_BOLD = find_font(["DejaVuSans-Bold.ttf"])
EMOJI_FONT = find_font(["NotoColorEmoji.ttf"])

_urdu_patterns = ["NotoNaskhArabic*Bold*.ttf", "NotoNaskhArabic*.ttf", "NotoSansArabic*Bold*.ttf",
                  "NotoSansArabic*.ttf", "Amiri-Bold.ttf", "Amiri*.ttf", "*Arab*.ttf"]
if USE_NASTALIQ:
  _urdu_patterns = ["NotoNastaliqUrdu*.ttf"] + _urdu_patterns
URDU_FONT = find_font(_urdu_patterns)
if URDU_FONT is None:
  raise RuntimeError("Urdu font nahi mila. Master Cell 1 dobara chalayein.")

HAS_RAQM = bool(features.check("raqm")) and bool(features.check("fribidi"))
LATIN_RE = re.compile(r"[A-Za-z]")
PUNCT_RE = re.compile(r"[۔؟،!?.,:;\"'()\-]")
COMMON_URDU = {"اور","ہے","ہیں","کو","میں","پر","کے","کی","کا","سے","یہ","وہ","تو","کہ","نہ","بھی","ہو","ہوتا",
               "ہوتی","ہوتے","کیا","آپ","ہم","جب","اس","ان","ایک","لیکن","ہر","کر","کرتے","لیں","رکھ","بار","دور",
               "والا","والے","والی","ہی","جو","کوئی","پھر","اب","تک","ہوں","تھا","تھی","گا","گی","گے","لیے"}

def norm_word(w: str) -> str:
  return PUNCT_RE.sub("", w).strip()

def urdu_prep(text: str):
  text = text.strip()
  if HAS_RAQM:
    return text, dict(direction="rtl", language="ur")
  return get_display(arabic_reshaper.reshape(text)), {}

_font_cache = {}
def F(path, size):
  key = (path, size)
  if key not in _font_cache:
    _font_cache[key] = ImageFont.truetype(path, size)
  return _font_cache[key]

_urdu_cache = {}
def U(size):
  if size not in _urdu_cache:
    engine = ImageFont.Layout.RAQM if HAS_RAQM else ImageFont.Layout.BASIC
    f = ImageFont.truetype(URDU_FONT, size, layout_engine=engine)
    try:
      f.set_variation_by_name("Bold")
    except Exception:
      pass
    _urdu_cache[size] = f
  return _urdu_cache[size]

def ease_out_cubic(x):
  x = min(1.0, max(0.0, x))
  return sub(1.0, pow(sub(1.0, x), 3))

def ease_out_back(x):
  x = min(1.0, max(0.0, x))
  c1 = 1.70158
  c3 = c1 + 1.0
  xm1 = sub(x, 1.0)
  return 1.0 + mul(c3, pow(xm1, 3)) + mul(c1, pow(xm1, 2))

def wrap_text(text, font, max_w):
  lines, cur = [], ""
  for w in text.split():
    test = (cur + " " + w).strip()
    if font.getlength(test) <= max_w:
      cur = test
    else:
      if cur:
        lines.append(cur)
      cur = w
  if cur:
    lines.append(cur)
  return lines

def fit_headline(text, max_w, max_lines=2):
  for size in (58, 54, 50, 46, 42, 38, 34):
    f = F(EN_BOLD, size)
    lines = wrap_text(text, f, max_w)
    if len(lines) <= max_lines and all(f.getlength(l) <= max_w for l in lines):
      return f, lines
  f = F(EN_BOLD, 32)
  return f, wrap_text(text, f, max_w)[:max_lines]

_emoji_cache = {}
def clean_emoji(e):
  e = (e or "").strip()
  if not e or all(ord(c) < 0x2000 for c in e):
    return "💡"
  out = e[0]
  prev_zwj = False
  for c in e[1:]:
    o = ord(c)
    if o in (0xFE0F, 0x200D) or 0x1F3FB <= o <= 0x1F3FF or prev_zwj:
      out += c
      prev_zwj = (o == 0x200D)
    else:
      break
  return out

def emoji_image(ch, size=256):
  key = (ch, size)
  if key in _emoji_cache:
    return _emoji_cache[key]
  img = None
  if EMOJI_FONT:
    try:
      ef = ImageFont.truetype(EMOJI_FONT, 109)
      tmp = Image.new("RGBA", (200, 200), (0, 0, 0, 0))
      ImageDraw.Draw(tmp).text((20, 20), ch, font=ef, embedded_color=True)
      bb = tmp.getbbox()
      if bb:
        crop = tmp.crop(bb)
        side = max(crop.size)
        sq = Image.new("RGBA", (side, side), (0, 0, 0, 0))
        sq.paste(crop, (sub(side, crop.width) // 2, sub(side, crop.height) // 2))
        img = sq.resize((size, size), Image.Resampling.LANCZOS)
    except Exception:
      img = None
  _emoji_cache[key] = img
  return img

# ==============================================================================
# 2. AUDIO ENVELOPE, AVATAR & SCENE BUILDER
# ==============================================================================
def get_audio_rms_envelope(wav_path: str, fps: int = VIDEO_FPS) -> np.ndarray:
  audio = AudioSegment.from_file(wav_path).set_channels(1).set_frame_rate(16000)
  samples = np.array(audio.get_array_of_samples(), dtype=np.float32)
  samples = np.abs(samples) / (np.max(np.abs(samples)) + 1e-6)
  samples_per_frame = int(16000 / fps)
  num_frames = int(math.ceil(len(samples) / samples_per_frame)) + 10
  env = np.zeros(num_frames, dtype=np.float32)
  for i in range(num_frames):
    s_idx = mul(i, samples_per_frame)
    e_idx = mul(i + 1, samples_per_frame)
    chunk = samples[s_idx:e_idx]
    if len(chunk) > 0:
      env[i] = float(np.sqrt(np.mean(np.square(chunk))))
  env = env / (np.percentile(env, 95) + 1e-6)
  return np.clip(env, 0.0, 1.0)

def prepare_circular_avatar(avatar_path: str, size: int = 150) -> Image.Image:
  if not os.path.exists(avatar_path):
    return None
  im = Image.open(avatar_path).convert("RGBA").resize((size, size), Image.Resampling.LANCZOS)
  mask = Image.new("L", (size, size), 0)
  ImageDraw.Draw(mask).ellipse((0, 0, size, size), fill=255)
  out = Image.new("RGBA", (size + 16, size + 16), (0, 0, 0, 0))
  draw = ImageDraw.Draw(out)
  r, g, b = ACCENT_MAIN
  bg_r, bg_g, bg_b = BG_MAIN
  draw.ellipse((0, 0, size + 15, size + 15), fill=(r, g, b, 255))
  draw.ellipse((5, 5, size + 10, size + 10), fill=(bg_r, bg_g, bg_b, 255))
  out.paste(im, (8, 8), mask)
  return out

def build_fast_scenes(words_list: list, target_duration: float = TARGET_SCENE_SEC, max_words: int = MAX_WORDS_PER_SCENE) -> list:
  scenes, cur_words = [], []
  for w in words_list:
    cur_words.append(w)
    dur = sub(cur_words[-1]["end"], cur_words[0]["start"])
    ends_punct = any(p in w["word"] for p in ["۔", "؟", "!", ".", "?"])
    if dur >= target_duration or len(cur_words) >= max_words or (ends_punct and dur >= 2.0):
      scenes.append({
          "scene_id": len(scenes),
          "start": cur_words[0]["start"],
          "end": cur_words[-1]["end"],
          "text": " ".join(x["word"] for x in cur_words),
          "words": cur_words,
      })
      cur_words = []
  if cur_words:
    scenes.append({
        "scene_id": len(scenes),
        "start": cur_words[0]["start"],
        "end": cur_words[-1]["end"],
        "text": " ".join(x["word"] for x in cur_words),
        "words": cur_words,
    })
  for i in range(sub(len(scenes), 1)):
    scenes[i]["end"] = scenes[i + 1]["start"]
  scenes[0]["start"] = 0.0
  return scenes

scenes_data = build_fast_scenes(words_data)

# ==============================================================================
# 3. AI DIRECTOR (Loads Directly from Fixed QWEN_DIR in Google Drive!)
# ==============================================================================
SCENE_FIELDS = ["badge", "headline", "emoji", "kind", "prompt"]

def _parse_json(resp: str) -> dict:
  try:
    m = re.search(r"\{.*\}", resp, re.DOTALL)
    return json.loads(m.group(0)) if m else {}
  except Exception:
    return {}

def generate_plan_with_qwen(scenes_list: list):
  print(f"🧠 Loading Qwen2.5-7B from Fixed Drive Vault: {QWEN_DIR}...")
  bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
  tokenizer = AutoTokenizer.from_pretrained(QWEN_DIR, local_files_only=True)
  model = AutoModelForCausalLM.from_pretrained(QWEN_DIR, quantization_config=bnb_config, device_map="auto", local_files_only=True)

  def ask(system, user, max_new=250, sample=True):
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    text_in = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text_in], return_tensors="pt").to(model.device)
    kw = dict(max_new_tokens=max_new)
    if sample:
      kw.update(do_sample=True, temperature=0.2)
    else:
      kw.update(do_sample=False)
    with torch.no_grad():
      out_ids = model.generate(**inputs, **kw)
    return tokenizer.decode(out_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

  full_transcript = " ".join([s["text"] for s in scenes_list])

  topic_data = _parse_json(ask(
      "You are a content analyzer.",
      "Analyze this Urdu/English transcript and output ONLY a valid JSON with keys:\n"
      " - 'main_topic': 2-4 words summarizing the core subject in punchy uppercase English.\n"
      " - 'visual_theme': 1 sentence describing the realistic documentary photography setting.\n\n"
      f"TRANSCRIPT: {full_transcript[:2000]}", 100, sample=False))
  main_topic = topic_data.get("main_topic", "SYSTEM DESIGN")
  visual_theme = topic_data.get("visual_theme", "Cinematic documentary photography")

  with open(OUTPUT_TOPIC_JSON, "w", encoding="utf-8") as f:
    json.dump({"main_topic": str(main_topic)}, f, ensure_ascii=False)

  sys_prompt = (
      f"You are an Editorial Documentary Visual Director for a short video about '{main_topic}'. Vibe: {visual_theme}.\n"
      "CRITICAL RULES:\n"
      "1. ALL visuals MUST look like real 35mm documentary photography (warm natural lighting, shallow depth of field, NO neon sci-fi AI glow, NO cartoons).\n"
      "2. Output ONLY valid JSON with keys:\n"
      "   - 'badge': 2-3 word English header.\n"
      "   - 'headline': a 3-5 word punchy English takeaway of the sentence, like 'CACHE = INSTANT MEMORY'.\n"
      "   - 'emoji': ONE single emoji that shows the meaning of the sentence.\n"
      "   - 'kind': one of 'tip', 'mistake', 'fix', 'fact'.\n"
      "   - 'prompt': a vivid 40-word realistic 35mm photography visual prompt."
  )

  prev_prompt = "None"
  for idx, s in enumerate(scenes_list):
    user_msg = f"CURRENT URDU SENTENCE: {s['text']}\nPREVIOUS VISUAL: {prev_prompt}\nOutput JSON:"
    parsed = _parse_json(ask(sys_prompt, user_msg, 250))

    s["badge"] = str(parsed.get("badge", f"PART {idx+1}")).upper()[:22]
    head = str(parsed.get("headline", main_topic)).upper().replace("\n", " ").strip()
    s["headline"] = " ".join(head.split()[:6])[:38] or str(main_topic).upper()
    s["emoji"] = clean_emoji(str(parsed.get("emoji", "")))
    kind = str(parsed.get("kind", "fact")).lower().strip()
    s["kind"] = kind if kind in ("tip", "mistake", "fix", "fact") else "fact"
    base_p = str(parsed.get("prompt", f"Editorial photo of {visual_theme}")).strip()
    prev_prompt = base_p[:90]
    s["prompt"] = f"{base_p}, shot on 35mm Kodak Portra 400 film, natural editorial lighting, shallow depth of field, documentary photography, ultra realistic"
    print(f"   ✨ Scene {idx+1:02d} [{s['kind'].upper()}] {s['emoji']} {s['headline']}")

  recap_raw = _parse_json(ask(
      "You write short takeaways for social media reels.",
      "From this Urdu/English transcript write the 3 most important takeaways in English. "
      "Each takeaway max 8 words, punchy. Output ONLY JSON: {\"recap\": [\"...\", \"...\", \"...\"]}\n\n"
      f"TRANSCRIPT: {full_transcript[:3000]}", 160, sample=False))
  recap = [str(x).strip() for x in recap_raw.get("recap", []) if str(x).strip()][:3]
  if len(recap) < 3:
    seen = []
    for s in scenes_list:
      if s["headline"] not in seen:
        seen.append(s["headline"])
    recap = (recap + seen)[:3]

  del model, tokenizer
  gc.collect()
  torch.cuda.empty_cache()

  with open(OUTPUT_PLAN_JSON, "w", encoding="utf-8") as f:
    json.dump(scenes_list, f, ensure_ascii=False, indent=2)
  with open(OUTPUT_RECAP_JSON, "w", encoding="utf-8") as f:
    json.dump(recap, f, ensure_ascii=False, indent=2)
  shutil.copy(OUTPUT_PLAN_JSON, os.path.join(PROJECT_DIR, OUTPUT_PLAN_JSON))
  shutil.copy(OUTPUT_RECAP_JSON, os.path.join(PROJECT_DIR, OUTPUT_RECAP_JSON))
  shutil.copy(OUTPUT_TOPIC_JSON, os.path.join(PROJECT_DIR, OUTPUT_TOPIC_JSON))
  return scenes_list, recap

def get_visual_plan(scenes_list: list):
  for fn in [OUTPUT_PLAN_JSON, OUTPUT_RECAP_JSON, OUTPUT_TOPIC_JSON]:
    if not os.path.exists(fn) and os.path.exists(os.path.join(PROJECT_DIR, fn)):
      shutil.copy(os.path.join(PROJECT_DIR, fn), fn)

  if not FORCE_REGEN and os.path.exists(OUTPUT_PLAN_JSON) and os.path.exists(OUTPUT_RECAP_JSON):
    try:
      with open(OUTPUT_PLAN_JSON, "r", encoding="utf-8") as f:
        cached = json.load(f)
      with open(OUTPUT_RECAP_JSON, "r", encoding="utf-8") as f:
        recap = json.load(f)
      topic_ok = (not SHOW_TITLE) or bool(TITLE_OVERRIDE.strip()) or os.path.exists(OUTPUT_TOPIC_JSON)
      ok = (len(cached) == len(scenes_list)
            and all(c.get("text") == s["text"] for c, s in zip(cached, scenes_list))
            and all(all(k in c for k in SCENE_FIELDS) for c in cached)
            and len(recap) >= 1
            and topic_ok)
      if ok:
        print("♻️ Cached visual plan loaded from Drive/Local! (Qwen skipped)")
        return cached, recap, True
    except Exception:
      pass
  scenes_out, recap = generate_plan_with_qwen(scenes_list)
  return scenes_out, recap, False

scenes_data, RECAP_ITEMS, plan_cached = get_visual_plan(scenes_data)

def load_topic() -> str:
  if TITLE_OVERRIDE.strip():
    return TITLE_OVERRIDE.strip().upper()
  try:
    with open(OUTPUT_TOPIC_JSON, "r", encoding="utf-8") as f:
      return json.load(f)["main_topic"].strip().upper()
  except Exception:
    return ""

TOPIC_TITLE = load_topic()
print("🏷️ Fixed Video Title :", TOPIC_TITLE or "(none)")

def build_english_map(words_list: list) -> dict:
  if not CONVERT_LOANWORDS:
    return dict(EN_OVERRIDES)
  if not os.path.exists(OUTPUT_EN_MAP_JSON) and os.path.exists(os.path.join(PROJECT_DIR, OUTPUT_EN_MAP_JSON)):
    shutil.copy(os.path.join(PROJECT_DIR, OUTPUT_EN_MAP_JSON), OUTPUT_EN_MAP_JSON)

  en_map = {}
  if os.path.exists(OUTPUT_EN_MAP_JSON) and not FORCE_REGEN:
    try:
      with open(OUTPUT_EN_MAP_JSON, "r", encoding="utf-8") as f:
        en_map = json.load(f)
    except Exception:
      en_map = {}

  uniq = []
  for w in words_list:
    k = norm_word(w["word"])
    if k and not LATIN_RE.search(k) and k not in COMMON_URDU and k not in en_map and k not in EN_OVERRIDES and k not in uniq:
      uniq.append(k)

  if uniq:
    print(f"🔤 Checking {len(uniq)} Urdu words for English loanwords...")
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
    tok = AutoTokenizer.from_pretrained(QWEN_DIR, local_files_only=True)
    mdl = AutoModelForCausalLM.from_pretrained(QWEN_DIR, quantization_config=bnb_config, device_map="auto", local_files_only=True)
    context = " ".join(w["word"] for w in words_list)[:2500]
    for i in range(0, len(uniq), 40):
      chunk = uniq[i:i + 40]
      sys_p = ("You fix a speech transcript where a speaker mixes Urdu and English. Some English words were "
               "written in Urdu letters. Find ONLY those English loanwords (example: کیش -> cache, ڈیٹا -> data, "
               "ویب -> web, سلو -> slow). Do NOT include real Urdu words. "
               "Output ONLY a valid JSON object mapping each loanword to its lowercase English spelling.")
      user_p = f"TRANSCRIPT CONTEXT: {context}\n\nWORDS TO CHECK: {json.dumps(chunk, ensure_ascii=False)}\nJSON:"
      msgs = [{"role": "system", "content": sys_p}, {"role": "user", "content": user_p}]
      text_in = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
      inputs = tok([text_in], return_tensors="pt").to(mdl.device)
      with torch.no_grad():
        out_ids = mdl.generate(**inputs, max_new_tokens=400, do_sample=False)
      resp = tok.decode(out_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
      found = _parse_json(resp)
      for k in chunk:
        v = str(found.get(k, "")).strip()
        en_map[k] = v.lower() if re.fullmatch(r"[A-Za-z][A-Za-z' ]*", v) else ""
    del mdl, tok
    gc.collect()
    torch.cuda.empty_cache()
    with open(OUTPUT_EN_MAP_JSON, "w", encoding="utf-8") as f:
      json.dump(en_map, f, ensure_ascii=False, indent=2)
    shutil.copy(OUTPUT_EN_MAP_JSON, os.path.join(PROJECT_DIR, OUTPUT_EN_MAP_JSON))

  final = {k: v for k, v in en_map.items() if v}
  final.update(EN_OVERRIDES)
  return final

EN_MAP = build_english_map([w for sc_ in scenes_data for w in sc_["words"]])

# ==============================================================================
# 4. IMAGE GENERATION (Uses Fixed LCM_DIR Directly + Warm Film Grading!)
# ==============================================================================
def generate_vibrant_images(scenes_list: list, reuse: bool = False):
  os.makedirs(OUTPUT_SCENE_IMG_DIR, exist_ok=True)
  drive_img_dir = os.path.join(PROJECT_DIR, OUTPUT_SCENE_IMG_DIR)
  os.makedirs(drive_img_dir, exist_ok=True)

  todo = []
  for s in scenes_list:
    fname = f"scene_{s['scene_id']:03d}.png"
    local_p = os.path.join(OUTPUT_SCENE_IMG_DIR, fname)
    drive_p = os.path.join(drive_img_dir, fname)
    if reuse and not os.path.exists(local_p) and os.path.exists(drive_p):
      shutil.copy(drive_p, local_p)
    s["image_path"] = local_p
    if not (reuse and os.path.exists(local_p)):
      todo.append(s)

  if not todo:
    print("\n♻️ All scene images loaded from cache! (Image generation skipped)")
    return

  print(f"\n🎨 Generating {len(todo)} Editorial 35mm Scene Images from {LCM_DIR}...")
  pipe = DiffusionPipeline.from_pretrained(
      LCM_DIR, torch_dtype=torch.float16, safety_checker=None, local_files_only=True
  ).to("cuda")
  neg_prompt = "cartoon, 3d, neon cyan glow, sci-fi, anime, illustration, painting, blurry, bad anatomy, text, watermark, deformed, oversaturated"

  for s in todo:
    seed_val = 1000 + mul(s["scene_id"], 42)
    generator = torch.Generator(device="cuda").manual_seed(seed_val)
    img = pipe(
        prompt=s["prompt"], negative_prompt=neg_prompt,
        width=768, height=768, num_inference_steps=8, guidance_scale=2.5, generator=generator
    ).images[0]
    # Natural Warm Documentary Film Grading (No artificial over-saturation)
    img = ImageEnhance.Color(img).enhance(1.04)
    img = ImageEnhance.Contrast(img).enhance(1.08)
    img.save(s["image_path"])
    shutil.copy(s["image_path"], os.path.join(drive_img_dir, os.path.basename(s["image_path"])))

  del pipe
  gc.collect()
  torch.cuda.empty_cache()

generate_vibrant_images(scenes_data, reuse=plan_cached)

# ==============================================================================
# 5. MULTI-SFX SYNTHESIZER (6 Distinct Studio Documentary Sound Effects!)
# ==============================================================================
main_audio = AudioSegment.from_file(AUDIO_PATH)
audio_len = len(main_audio) / 1000.0
scenes_data[-1]["end"] = min(audio_len, scenes_data[-1]["end"] + 0.5)
CONTENT_END = scenes_data[-1]["end"]
TOTAL_DUR = CONTENT_END + RECAP_DUR

n_tips = sum(1 for s in scenes_data if s["kind"] == "tip")
tip_i = 0
for s in scenes_data:
  if s["kind"] == "tip":
    tip_i += 1
    s["tag"] = f"TIP {tip_i}/{n_tips}" if n_tips > 1 else "TIP"
  else:
    s["tag"] = s["kind"].upper()

def _save_wave(filename, wave_arr, sr=44100, peak=0.8):
  wave_arr = wave_arr / (np.max(np.abs(wave_arr)) + 1e-6)
  scaled = np.int16(mul(mul(wave_arr, peak), 32767.0))
  wavfile.write(filename, sr, scaled)
  return AudioSegment.from_file(filename)

def build_multi_sfx_bank():
  sr = 44100
  # Transition 1: Soft Air Swish
  t1 = np.linspace(0, 0.32, int(mul(sr, 0.32)), False)
  env1 = np.power(np.sin(mul(np.pi, t1 / 0.32)), 2)
  swish = mul(np.sin(mul(mul(2.0, np.pi), mul(np.linspace(650, 140, len(t1)), t1))), 0.4) + mul(np.random.normal(0, 0.25, len(t1)), env1)
  tr_swish = _save_wave("sfx_tr1.wav", mul(swish, env1), sr) + SFX_TRANSITION_VOL_DB

  # Transition 2: Deep Cinematic Sub Thud
  t2 = np.linspace(0, 0.28, int(mul(sr, 0.28)), False)
  env2 = np.exp(mul(-14.0, t2))
  thud = mul(np.sin(mul(mul(2.0, np.pi), mul(np.linspace(130, 42, len(t2)), t2))), env2)
  tr_thud = _save_wave("sfx_tr2.wav", thud, sr) + SFX_TRANSITION_VOL_DB

  # Transition 3: Warm Tape Whoosh
  t3 = np.linspace(0, 0.36, int(mul(sr, 0.36)), False)
  env3 = np.power(np.sin(mul(np.pi, t3 / 0.36)), 1.8)
  tape = mul(np.sin(mul(mul(2.0, np.pi), mul(np.linspace(320, 780, len(t3)), t3))), env3)
  tr_tape = _save_wave("sfx_tr3.wav", tape, sr) + SFX_TRANSITION_VOL_DB

  # Accent 1: Subtle Mechanical Key Tick
  ta1 = np.linspace(0, 0.06, int(mul(sr, 0.06)), False)
  tick = mul(np.sin(mul(mul(2.0, np.pi), mul(1800.0, ta1))) + np.random.normal(0, 0.3, len(ta1)), np.exp(mul(-65.0, ta1)))
  ac_tick = _save_wave("sfx_ac1.wav", tick, sr) + SFX_ACCENT_VOL_DB

  # Accent 2: Warm Wooden Marimba Tap
  ta2 = np.linspace(0, 0.11, int(mul(sr, 0.11)), False)
  wood = mul(np.sin(mul(mul(2.0, np.pi), mul(587.33, ta2))) + mul(0.3, np.sin(mul(mul(2.0, np.pi), mul(1174.66, ta2)))), np.exp(mul(-36.0, ta2)))
  ac_wood = _save_wave("sfx_ac2.wav", wood, sr) + SFX_ACCENT_VOL_DB

  # Accent 3: Soft Glass Harmonic Pluck
  ta3 = np.linspace(0, 0.14, int(mul(sr, 0.14)), False)
  pluck = mul(np.sin(mul(mul(2.0, np.pi), mul(783.99, ta3))) + mul(0.4, np.sin(mul(mul(2.0, np.pi), mul(1567.98, ta3)))), np.exp(mul(-28.0, ta3)))
  ac_pluck = _save_wave("sfx_ac3.wav", pluck, sr) + SFX_ACCENT_VOL_DB

  # Outro: Warm Acoustic Major-7th Chord Chime
  tc = np.linspace(0, 1.1, int(mul(sr, 1.1)), False)
  chord = (np.sin(mul(mul(2.0, np.pi), mul(523.25, tc)))
           + mul(0.7, np.sin(mul(mul(2.0, np.pi), mul(659.25, tc))))
           + mul(0.5, np.sin(mul(mul(2.0, np.pi), mul(783.99, tc))))
           + mul(0.4, np.sin(mul(mul(2.0, np.pi), mul(987.77, tc)))))
  chime = _save_wave("sfx_chime.wav", mul(chord, np.exp(mul(-3.8, tc))), sr) + SFX_OUTRO_CHIME_VOL_DB

  return [tr_swish, tr_thud, tr_tape], [ac_tick, ac_wood, ac_pluck], chime

print("\n🔊 Mixing Varied Editorial Sound Effects...")
main_audio = main_audio[: int(mul(CONTENT_END, 1000))]
main_audio = main_audio + AudioSegment.silent(duration=int(mul(RECAP_DUR, 1000)), frame_rate=main_audio.frame_rate).set_channels(main_audio.channels)

if ENABLE_SFX:
  transitions_sfx, accents_sfx, outro_chime = build_multi_sfx_bank()
  for s in scenes_data:
    sid = s["scene_id"]
    cut_ms = int(mul(s["start"], 1000))
    if sid > 0:
      # Rotate through the 3 different transition sounds so it never sounds repetitive!
      tr_sound = transitions_sfx[sid % len(transitions_sfx)]
      main_audio = main_audio.overlay(tr_sound, position=cut_ms)
    # Rotate through the 3 different subtle UI accent sounds based on scene type/index
    ac_sound = accents_sfx[(sid + len(s["kind"])) % len(accents_sfx)]
    main_audio = main_audio.overlay(ac_sound, position=cut_ms + 110)
  main_audio = main_audio.overlay(outro_chime, position=int(mul(CONTENT_END, 1000)) + 100)

main_audio.export(OUTPUT_MIXED_AUDIO_WAV, format="wav")

# ==============================================================================
# 6. LAYOUT HELPERS (Urdu Karaoke, Fixed Top Title, Stickers & Cards)
# ==============================================================================
URDU_TOP, URDU_BOTTOM = 1380, 1745

def layout_urdu(words, max_w=900, gap=28):
  for size in (60, 54, 48, 42, 38):
    font = U(size)
    ref_txt, ref_kw = urdu_prep("آپگٹ")
    rb = font.getbbox(ref_txt, **ref_kw)
    rh = sub(rb[3], rb[1])
    line_h = rh + 56
    items = []
    for w in words:
      raw = w["word"].strip()
      if not raw:
        continue
      key = norm_word(raw)
      if not LATIN_RE.search(raw) and key in EN_MAP:
        raw = EN_MAP[key]
      if LATIN_RE.search(raw):
        lt = PUNCT_RE.sub("", raw).strip().upper()
        lf = F(EN_BOLD, int(mul(size, 0.74)))
        bb = lf.getbbox(lt)
        items.append(dict(txt=lt, kw={}, font=lf, lat=True, cy=(bb[1] + bb[3]) / 2.0,
                          w=max(1, sub(bb[2], bb[0])), ox=bb[0], start=w["start"], end=w["end"]))
      else:
        shaped, kw = urdu_prep(raw)
        bb = font.getbbox(shaped, **kw)
        items.append(dict(txt=shaped, kw=kw, font=font, lat=False,
                          w=max(1, sub(bb[2], bb[0])), ox=bb[0], start=w["start"], end=w["end"]))
    lines, cur, cw = [], [], 0
    for it in items:
      add = it["w"] + (gap if cur else 0)
      if cur and cw + add > max_w:
        lines.append(cur)
        cur, cw = [it], it["w"]
      else:
        cur.append(it)
        cw += add
    if cur:
      lines.append(cur)
    block = mul(len(lines), line_h)
    if (len(lines) <= 3 and block <= sub(URDU_BOTTOM, URDU_TOP)) or size == 38:
      break

  y0 = URDU_TOP + max(0, sub(sub(URDU_BOTTOM, URDU_TOP), block) // 2)
  for li, line in enumerate(lines):
    total = sum(i["w"] for i in line) + mul(gap, sub(len(line), 1))
    x = W / 2.0 + total / 2.0
    line_top = y0 + mul(li, line_h)
    for it in line:
      it["x"] = sub(x, it["w"])
      x = sub(it["x"], gap)
      it["pill_top"] = sub(line_top + 28, 12)
      it["pill_bot"] = line_top + 28 + rh + 12
      if it["lat"]:
        it["draw_y"] = sub((it["pill_top"] + it["pill_bot"]) / 2.0, it["cy"])
      else:
        it["draw_y"] = sub(line_top + 28, rb[1])
  flat = [it for line in lines for it in line]
  return font, flat

_prep_cache = {}
def prep_scene(sc):
  sid = sc["scene_id"]
  if sid in _prep_cache:
    return _prep_cache[sid]
  head_font, head_lines = fit_headline(sc["headline"].upper(), max_w=734, max_lines=2)
  ufont, uitems = layout_urdu(sc["words"])
  _prep_cache[sid] = dict(head_font=head_font, head_lines=head_lines, ufont=ufont, uitems=uitems)
  return _prep_cache[sid]

def draw_sticker(frame, sc, t):
  if not SHOW_EMOJI_STICKER:
    return
  el = sub(t, sc["start"])
  if el < 0:
    return
  img = emoji_image(sc["emoji"]) or emoji_image("💡")
  if img is None:
    return
  scale = ease_out_back(min(1.0, el / 0.45))
  size = int(mul(210, scale))
  if size < 10:
    return
  im = img.resize((size, size), Image.Resampling.LANCZOS)
  ang = mul(mul(12.0, math.sin(mul(el, 8.0))), math.exp(mul(-3.2, el)))
  im = im.rotate(ang, expand=True, resample=Image.Resampling.BICUBIC)
  bob = mul(5.0, math.sin(mul(mul(2.0, math.pi), mul(el, 0.8))))
  cx, cy = sub(W, 165), 165 + bob
  layer = Image.new("RGBA", (W, 400), (0, 0, 0, 0))
  d = ImageDraw.Draw(layer)
  r = mul(size, 0.58)
  d.ellipse([sub(cx, r) + 4, sub(cy, r) + 8, cx + r + 4, cy + r + 8], fill=(0, 0, 0, 95))
  d.ellipse([sub(cx, r), sub(cy, r), cx + r, cy + r], fill=(255, 255, 255, 48))
  layer.paste(im, (int(sub(cx, im.width / 2.0)), int(sub(cy, im.height / 2.0))), im)
  frame.paste(layer, (0, 0), layer)

# ---- ROCK-SOLID FIXED TOPIC TITLE BANNER (No sliding/bouncing) ----
_title_cache = {}
def draw_title_fixed(frame, t):
  if not SHOW_TITLE or not TOPIC_TITLE or t > CONTENT_END:
    return
  if "layer" not in _title_cache:
    f, lines = fit_headline(TOPIC_TITLE, max_w=680, max_lines=2)
    lh = int(mul(f.size, 1.2))
    box_w = max(int(f.getlength(l)) for l in lines) + 70
    box_h = mul(len(lines), lh) + 34
    x0, y0 = 40, 48
    layer = Image.new("RGBA", (W, 260), (0, 0, 0, 0))
    d = ImageDraw.Draw(layer)
    d.rounded_rectangle([x0, y0, x0 + box_w, y0 + box_h], radius=22,
                        fill=TITLE_BG_RGBA, outline=TITLE_BORDER_RGBA, width=3)
    ty = y0 + 17
    for ln in lines:
      d.text((x0 + 35, ty), ln, font=f, fill=TITLE_TEXT_RGBA)
      ty += lh
    _title_cache["layer"] = layer
  frame.paste(_title_cache["layer"], (0, 0), _title_cache["layer"])

def draw_save_nudge(draw, t):
  t0 = mul(SAVE_NUDGE_AT, CONTENT_END)
  el = sub(t, t0)
  dur = 3.2
  if el < 0 or el > dur:
    return
  if el < 0.4:
    off = int(mul(sub(1.0, ease_out_back(el / 0.4)), -760.0))
  elif el > sub(dur, 0.3):
    off = int(mul(-760.0, ease_out_cubic(sub(el, sub(dur, 0.3)) / 0.3)))
  else:
    off = 0
  x = 40 + off
  draw.rounded_rectangle([x, 225, x + 690, 312], radius=24, fill=ACCENT_MAIN, outline=(255, 255, 255), width=3)
  draw.polygon([(x + 34, 245), (x + 68, 245), (x + 68, 296), (x + 51, 281), (x + 34, 296)], fill=TEXT_ON_ACCENT)
  draw.text((x + 92, 251), SAVE_NUDGE_TEXT, font=F(EN_BOLD, 36), fill=TEXT_ON_ACCENT)

def draw_recap(frame, t, items):
  el = sub(t, CONTENT_END)
  if el < 0:
    return frame
  a = ease_out_cubic(el / 0.35)
  frame = Image.blend(frame, Image.new("RGB", (W, H), BG_MAIN), mul(0.88, a))
  draw = ImageDraw.Draw(frame)
  slide = int(mul(sub(1.0, ease_out_cubic(el / 0.55)), 220.0))
  x0, x1 = 60, sub(W, 60)
  y0 = 520 + slide
  y1 = y0 + 880
  draw.rounded_rectangle([x0, y0, x1, y1], radius=40, fill=CARD_BG, outline=ACCENT_MAIN, width=4)

  tf = F(EN_BOLD, 68)
  tw = tf.getlength(RECAP_HEADER_TEXT)
  draw.text((sub(W, tw) / 2.0, y0 + 48), RECAP_HEADER_TEXT, font=tf, fill=ACCENT_MAIN)
  draw.line([(x0 + 80, y0 + 155), (sub(x1, 80), y0 + 155)], fill=DIVIDER_COLOR, width=3)

  rf = F(EN_BOLD, 44)
  for i, item in enumerate(items[:3]):
    if el < 0.45 + mul(0.42, i):
      continue
    ry = y0 + 195 + mul(i, 195)
    ccx, ccy = x0 + 95, ry + 62
    draw.ellipse([sub(ccx, 38), sub(ccy, 38), ccx + 38, ccy + 38], fill=ACCENT_MAIN)
    draw.line([(sub(ccx, 16), ccy + 2), (sub(ccx, 5), ccy + 14), (ccx + 18, sub(ccy, 13))],
              fill=TEXT_ON_ACCENT, width=8, joint="curve")
    max_line_w = sub(sub(x1, 50), x0 + 170)
    lines = wrap_text(item.upper(), rf, max_line_w)[:2]
    ty = sub(ccy, mul(len(lines), 56) // 2)
    for ln in lines:
      draw.text((x0 + 170, ty), ln, font=rf, fill=HEADLINE_COLOR)
      ty += 56

  ff = F(EN_BOLD, 34)
  draw.text((sub(W, ff.getlength(FOLLOW_CTA_TEXT)) / 2.0, sub(y1, 82)), FOLLOW_CTA_TEXT, font=ff, fill=ACCENT_MAIN)
  return frame

# ==============================================================================
# 7. RENDER FINAL EDITORIAL REEL
# ==============================================================================
def build_final_split_screen_reel(scenes_list: list, audio_path: str, output_mp4: str = OUTPUT_VIDEO_MP4, fps: int = VIDEO_FPS):
  font_badge = F(EN_BOLD, 34)
  font_tag = F(EN_BOLD, 30)
  font_wm = F(EN_BOLD, 28)
  font_name = F(EN_BOLD, 22)

  AV = 150
  avatar_img = prepare_circular_avatar(AVATAR_PATH, size=AV)
  voice_env = get_audio_rms_envelope(audio_path, fps=fps)

  loaded_imgs = {s["scene_id"]: Image.open(s["image_path"]).convert("RGB") for s in scenes_list}

  grad = Image.new("RGBA", (W, 130), (0, 0, 0, 0))
  gd = ImageDraw.Draw(grad)
  bg_r, bg_g, bg_b = BG_MAIN
  for gy in range(130):
    alpha = int(mul(255.0, pow(gy / 129.0, 1.5)))
    gd.line([(0, gy), (W, gy)], fill=(bg_r, bg_g, bg_b, alpha))

  total_frames = int(math.ceil(mul(TOTAL_DUR, fps)))
  print(f"\n🎬 Rendering Editorial Reel ({total_frames} frames) -> {output_mp4}...")

  ffmpeg_cmd = [
      "ffmpeg", "-y", "-f", "rawvideo", "-vcodec", "rawvideo", "-s", f"{W}x{H}",
      "-pix_fmt", "rgb24", "-r", str(fps), "-i", "-", "-i", audio_path,
      "-c:v", "libx264", "-preset", "fast", "-crf", "18", "-pix_fmt", "yuv420p",
      "-c:a", "aac", "-b:a", "192k", "-shortest", output_mp4,
  ]
  proc = subprocess.Popen(ffmpeg_cmd, stdin=subprocess.PIPE, stderr=subprocess.DEVNULL)

  scene_idx = 0
  for f_idx in range(total_frames):
    t = f_idx / float(fps)
    while scene_idx < sub(len(scenes_list), 1) and t >= scenes_list[scene_idx]["end"]:
      scene_idx += 1
    sc = scenes_list[scene_idx]
    pr = prep_scene(sc)

    # ---------- Top Picture with Smooth Documentary Ken-Burns ----------
    sc_dur = max(0.15, sub(sc["end"], sc["start"]))
    prog = min(1.0, max(0.0, sub(t, sc["start"]) / sc_dur))
    motion_type = sc["scene_id"] % 4
    src_img = loaded_imgs[sc["scene_id"]]
    iw, ih = src_img.size
    if motion_type == 0:
      zoom = 1.04 + mul(0.12, prog)
    elif motion_type == 1:
      zoom = sub(1.16, mul(0.12, prog))
    else:
      zoom = 1.14
    crop_w = int(iw / zoom)
    crop_h = min(ih, int(mul(crop_w, TOP_H) / float(W)))
    top = sub(ih, crop_h) // 2
    if motion_type == 2:
      left = int(mul(sub(iw, crop_w), prog))
    elif motion_type == 3:
      left = int(mul(sub(iw, crop_w), sub(1.0, prog)))
    else:
      left = sub(iw, crop_w) // 2
    top_visual = src_img.crop((left, top, left + crop_w, top + crop_h)).resize((W, TOP_H), Image.Resampling.BILINEAR)

    frame = Image.new("RGB", (W, H), BG_MAIN)
    frame.paste(top_visual, (0, 0))
    frame.paste(grad, (0, sub(TOP_H, 130)), grad)

    # ---------- Emoji Sticker & Fixed Topic Title ----------
    draw_sticker(frame, sc, t)
    draw_title_fixed(frame, t)
    draw = ImageDraw.Draw(frame)

    # ---------- Editorial Divider Line + Scene Badge ----------
    draw.rectangle([0, sub(TOP_H, 3), W, TOP_H + 3], fill=ACCENT_MAIN)
    bt = sc["badge"]
    bb = draw.textbbox((0, 0), bt, font=font_badge)
    bw = sub(bb[2], bb[0]) + 86
    bh = sub(bb[3], bb[1]) + 26
    bx, by = 55, sub(TOP_H, bh // 2)
    draw.rounded_rectangle([bx, by, bx + bw, by + bh], radius=18, fill=ACCENT_MAIN, outline=BG_MAIN, width=3)
    draw.ellipse([bx + 22, sub(by + bh // 2, 7), bx + 36, by + bh // 2 + 7], fill=TEXT_ON_ACCENT)
    draw.text((bx + 50, sub(by + 10, bb[1])), bt, font=font_badge, fill=TEXT_ON_ACCENT)

    # ---------- Matte Editorial Studio Card ----------
    draw.rounded_rectangle([40, TOP_H + 55, sub(W, 40), sub(H, 70)], radius=34, fill=CARD_BG, outline=CARD_BORDER, width=2)

    # Tag Pill (TIP, MISTAKE, FIX, FACT)
    tcol = TAG_COLORS.get(sc["kind"], ACCENT_MAIN)
    tb = draw.textbbox((0, 0), sc["tag"], font=font_tag)
    tw = sub(tb[2], tb[0])
    draw.rounded_rectangle([80, TOP_H + 80, 80 + tw + 42, TOP_H + 128], radius=14, fill=tcol)
    draw.text((101, sub(TOP_H + 104, (tb[1] + tb[3]) // 2)), sc["tag"], font=font_tag, fill=TEXT_ON_ACCENT)

    # Headline (Warm Cream / High Contrast Editorial Typography)
    hy = TOP_H + 144
    hf = pr["head_font"]
    for ln in pr["head_lines"]:
      draw.text((80, hy), ln, font=hf, fill=HEADLINE_COLOR)
      hy += int(mul(hf.size, 1.18))

    # Voice-Reactive Studio Avatar
    if avatar_img is not None:
      vol = float(voice_env[min(f_idx, sub(len(voice_env), 1))])
      pulse = int(3 + mul(10.0, vol))
      ax, ay = 844, TOP_H + 78
      cx, cy = ax + (AV + 16) // 2, ay + (AV + 16) // 2
      rr = (AV + 16) // 2
      draw.ellipse([sub(sub(cx, rr), pulse), sub(sub(cy, rr), pulse), cx + rr + pulse, cy + rr + pulse],
                   outline=ACCENT_MAIN, width=4)
      frame.paste(avatar_img, (ax, ay), avatar_img)
      nw = font_name.getlength(PRESENTER_NAME_TEXT)
      ny = ay + AV + 26
      draw.rounded_rectangle([sub(cx - nw / 2.0, 18), ny, cx + nw / 2.0 + 18, ny + 38], radius=12, fill=ACCENT_MAIN)
      draw.text((sub(cx, nw / 2.0), ny + 6), PRESENTER_NAME_TEXT, font=font_name, fill=TEXT_ON_ACCENT)

    draw.line([(80, TOP_H + 310), (sub(W, 80), TOP_H + 310)], fill=DIVIDER_COLOR, width=2)

    # ---------- Urdu Karaoke (Warm Gold Active Pill + Crisp Cream Text) ----------
    active_idx = -1
    for wi, it in enumerate(pr["uitems"]):
      if t >= it["start"]:
        active_idx = wi
    for wi, it in enumerate(pr["uitems"]):
      px = sub(it["x"], it["ox"])
      if wi == active_idx:
        draw.rounded_rectangle([sub(it["x"], 14), it["pill_top"], it["x"] + it["w"] + 14, it["pill_bot"]],
                               radius=16, fill=ACCENT_MAIN)
        draw.text((px, it["draw_y"]), it["txt"], font=it["font"], fill=TEXT_ON_ACCENT, **it["kw"])
      elif wi < active_idx:
        draw.text((px, it["draw_y"]), it["txt"], font=it["font"], fill=URDU_SPOKEN, **it["kw"])
      else:
        draw.text((px, it["draw_y"]), it["txt"], font=it["font"], fill=URDU_UPCOMING, **it["kw"])

    # ---------- Save Nudge, Watermark & Progress Bar ----------
    draw_save_nudge(draw, t)
    draw.text((sub(W, font_wm.getlength(WATERMARK_TEXT)) / 2.0, sub(H, 160)),
              WATERMARK_TEXT, font=font_wm, fill=WATERMARK_COLOR)
    bar_w = int(mul(sub(W, 160), min(1.0, t / TOTAL_DUR)))
    draw.rounded_rectangle([80, sub(H, 100), sub(W, 80), sub(H, 86)], radius=7, fill=PROGRESS_BG)
    if bar_w > 0:
      draw.rounded_rectangle([80, sub(H, 100), 80 + bar_w, sub(H, 86)], radius=7, fill=ACCENT_MAIN)

    # ---------- Recap Card at the End ----------
    frame = draw_recap(frame, t, RECAP_ITEMS)
    proc.stdin.write(frame.tobytes())

  proc.stdin.close()
  proc.wait()

  drive_out = os.path.join(PROJECT_DIR, output_mp4)
  shutil.copy(output_mp4, drive_out)
  print(f"\n🎉 FINAL EDITORIAL REEL SAVED TO DRIVE: {drive_out}")
  return output_mp4

final_mp4 = build_final_split_screen_reel(scenes_data, OUTPUT_MIXED_AUDIO_WAV)

# 62914560 bytes = 60 MB
if os.path.getsize(final_mp4) < 62914560:
  with open(final_mp4, "rb") as f:
    b64_vid = b64encode(f.read()).decode("utf-8")
  display(HTML(f'<video width="380" controls style="border-radius:14px; border:2px solid #F5B73B;"><source src="data:video/mp4;base64,{b64_vid}" type="video/mp4"></video>'))
else:
  print("Video bari hai, preview skip. Drive se dekh lein.")